#### **Name(s):** Ibrahim Labib, Islam Akram and Shahd Kazan
#### **ID(s):** 232400591, 232400623 and 232400237

# ***MLOps Final on Telco Churn*** - MLflow & Training

In [11]:
# ============================================================
# MLOps Final Project — Ibrahim's Part
# One-cell Colab notebook version
# ============================================================
#
# This single cell:
# 1. Installs required packages.
# 2. Creates the project folder structure.
# 3. Uploads and places the required files manually in Colab.
# 4. Creates all source files for Ibrahim's Training + DVC + MLflow part.
# 5. Trains the baseline model.
# 6. Logs MLflow experiments using all features.
# 7. Runs feature importance analysis as a comparison, not as an automatic replacement.
# 8. Keeps the all-feature model as final if feature selection does not improve performance.
# 9. Runs hyperparameter tuning.
# 10. Registers the best model in MLflow.
# 11. Updates Shahd's existing dvc.yaml by appending Ibrahim's stages only.
# 12. Exports one ZIP file per GitHub issue.
#
# Correct GitHub issue mapping:
# #16 — DVC pipeline stages
# #17 — Train baseline model
# #18 — Model evaluation
# #19 — Setup MLflow
# #20 — Log experiments
# #21 — Hyperparameter tuning
# #22 — Register best model
#
# Important:
# - This cell does NOT include GitHub commands.
# - This Colab notebook is NOT connected to GitHub.
# - You manually upload the data and required project files into Colab.
# - The generated ZIP files are downloaded and committed later from the local repo.
# - Do NOT create new prepare.py, preprocess.py, or feature-engineering scripts.
# - Shahd owns prepare/features/preprocess in the shared DVC pipeline.
# - Issue #16 uses Shahd's uploaded dvc.yaml and appends only Ibrahim's stages.
# - Data drift detection has been removed.
# - Feature importance is kept as analysis unless it beats the all-feature model.
# ============================================================

### **Workflow Setup and Libraries**

In [12]:
# ============================================================
# 0. Install required dependencies
# ============================================================
#
# Purpose:
# - Install MLflow for experiment tracking and model registry.
# - Install DVC for pipeline metadata.
# - Install sklearn, pandas, numpy, joblib, scipy, yaml tools.
# - Install XGBoost, LightGBM, and CatBoost for boosting model comparison.
#
# Notes:
# - No GitHub commands are used here.
# - This is safe for Colab because everything is installed inside the runtime.
# ============================================================

import sys
import subprocess

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "mlflow",
        "dvc",
        "scikit-learn",
        "pandas",
        "numpy",
        "joblib",
        "pyyaml",
        "scipy",
        "xgboost",
        "lightgbm",
        "catboost",
    ],
    check=True,
)


# ============================================================
# 1. Import standard libraries after installation
# ============================================================
#
# Purpose:
# - Import file-system utilities.
# - Import JSON, ZIP, YAML, and pandas.
# - These imports are used throughout the one-cell notebook.
# ============================================================

import os
import json
import shutil
import zipfile
from pathlib import Path

import pandas as pd
import yaml


# ============================================================
# 2. Create a clean Colab workspace
# ============================================================
#
# Purpose:
# - Create a clean isolated folder for the generated project.
# - Remove previous Colab outputs to prevent stale files from causing errors.
# - Create the same folder structure expected by the repository.
#
# Output:
# - /content/ibrahim_mlops_project
# - src/
# - src/data/
# - src/training/
# - src/evaluation/
# - configs/
# - data/
# - models/
# - reports/
# - mlruns/
# - exports/
# ============================================================

PROJECT_ROOT = Path("/content/ibrahim_mlops_project")

if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_ROOT)

os.environ["PYTHONPATH"] = f"{PROJECT_ROOT}:{os.environ.get('PYTHONPATH', '')}"

folders_to_create = [
    "src",
    "src/data",
    "src/training",
    "src/evaluation",
    "configs",
    "data",
    "data/raw",
    "data/splits",
    "data/processed",
    "data/features",
    "models",
    "reports",
    "reports/pipeline",
    "reports/feature_importance",
    "mlruns",
    "exports",
]

for folder in folders_to_create:
    Path(folder).mkdir(parents=True, exist_ok=True)

Path("src/__init__.py").touch()
Path("src/data/__init__.py").touch()
Path("src/training/__init__.py").touch()
Path("src/evaluation/__init__.py").touch()

print("Workspace created at:", PROJECT_ROOT)


# ============================================================
# 3. Upload real project data and existing shared pipeline files
# ============================================================
#
# Purpose:
# - Manually upload files into Colab.
# - This is required because the notebook is not connected to GitHub.
#
# Required uploads:
# - train.csv
# - test.csv
# - WA_Fn-UseC_-Telco-Customer-Churn.csv.dvc
# - prepare.py
# - preprocess.py
# - dvc.yaml
# - params.yaml
#
# Important:
# - dvc.yaml must be Shahd's latest shared version.
# - params.yaml must be Shahd's latest configs/params.yaml.
# - prepare.py and preprocess.py must be Shahd's existing repo files.
# - This notebook will NOT create replacement prepare/preprocess files.
# - This notebook will NOT run dvc pull inside Colab.
# ============================================================

try:
    from google.colab import files

    print("\nUpload these files when prompted:")
    print("- train.csv")
    print("- test.csv")
    print("- WA_Fn-UseC_-Telco-Customer-Churn.csv.dvc")
    print("- prepare.py")
    print("- preprocess.py")
    print("- dvc.yaml")
    print("- params.yaml")

    uploaded = files.upload()

    for uploaded_file in uploaded.keys():
        print("Uploaded:", uploaded_file)

except Exception:
    print("Not running in Colab or upload tool unavailable.")
    print("The script will look for files in the current directory.")


# ============================================================
# 4. Move uploaded files into the required project structure
# ============================================================
#
# Purpose:
# - Validate all required uploaded files.
# - Place train/test data under data/splits/.
# - Place the raw DVC pointer under data/raw/.
# - Place Shahd's prepare.py and preprocess.py under src/data/.
# - Place Shahd's params.yaml under configs/.
# - Keep Shahd's dvc.yaml at the project root.
#
# Important:
# - dvc.yaml may already be in the project root after upload.
# - params.yaml may already be in the project root after upload.
# - The code avoids copying files onto themselves to prevent SameFileError.
# ============================================================

train_source = Path("train.csv")
test_source = Path("test.csv")
raw_dvc_source = Path("WA_Fn-UseC_-Telco-Customer-Churn.csv.dvc")
prepare_source = Path("prepare.py")
preprocess_source = Path("preprocess.py")
dvc_yaml_source = Path("dvc.yaml")
params_source = Path("params.yaml")

if not train_source.exists():
    raise FileNotFoundError("Missing train.csv. Upload or place it in the current directory.")

if not test_source.exists():
    raise FileNotFoundError("Missing test.csv. Upload or place it in the current directory.")

if not raw_dvc_source.exists():
    raise FileNotFoundError(
        "Missing WA_Fn-UseC_-Telco-Customer-Churn.csv.dvc. "
        "Upload or place it in the current directory."
    )

if not prepare_source.exists():
    raise FileNotFoundError(
        "Missing prepare.py. Upload the existing src/data/prepare.py file from the repo."
    )

if not preprocess_source.exists():
    raise FileNotFoundError(
        "Missing preprocess.py. Upload the existing src/data/preprocess.py file from the repo."
    )

if not dvc_yaml_source.exists():
    raise FileNotFoundError(
        "Missing dvc.yaml. Upload Shahd's latest dvc.yaml from the repo."
    )

if not params_source.exists():
    raise FileNotFoundError(
        "Missing params.yaml. Upload Shahd's latest configs/params.yaml from the repo."
    )

shutil.copy2(train_source, "data/splits/train.csv")
shutil.copy2(test_source, "data/splits/test.csv")
shutil.copy2(raw_dvc_source, "data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv.dvc")

shutil.copy2(prepare_source, "src/data/prepare.py")
shutil.copy2(preprocess_source, "src/data/preprocess.py")

dvc_yaml_destination = Path("dvc.yaml")

if dvc_yaml_source.resolve() != dvc_yaml_destination.resolve():
    shutil.copy2(dvc_yaml_source, dvc_yaml_destination)
else:
    print("dvc.yaml is already in the project root. Skipping copy.")

params_destination = Path("configs/params.yaml")
params_destination.parent.mkdir(parents=True, exist_ok=True)

if params_source.resolve() != params_destination.resolve():
    shutil.copy2(params_source, params_destination)
else:
    print("params.yaml is already in configs/. Skipping copy.")

print("\nFiles placed successfully:")
print("- data/splits/train.csv")
print("- data/splits/test.csv")
print("- data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv.dvc")
print("- src/data/prepare.py")
print("- src/data/preprocess.py")
print("- dvc.yaml")
print("- configs/params.yaml")


# ============================================================
# 5. Inspect the real data and detect the target column
# ============================================================
#
# Purpose:
# - Load train/test data.
# - Detect the target column automatically.
# - Support both dummy data using target and Telco data using Churn.
#
# Output:
# - TARGET_COLUMN variable used by all generated scripts.
# ============================================================

train_df = pd.read_csv("data/splits/train.csv")
test_df = pd.read_csv("data/splits/test.csv")

if "target" in train_df.columns:
    TARGET_COLUMN = "target"
elif "Churn" in train_df.columns:
    TARGET_COLUMN = "Churn"
else:
    TARGET_COLUMN = train_df.columns[-1]

print("\nDataset inspection:")
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Detected target column:", TARGET_COLUMN)
print("Columns:", train_df.columns.tolist())

print("\nTrain target distribution:")
print(train_df[TARGET_COLUMN].value_counts())

print("\nTest target distribution:")
print(test_df[TARGET_COLUMN].value_counts())


# ============================================================
# 6. Helper function to write Python files cleanly
# ============================================================
#
# Purpose:
# - Create source files programmatically.
# - Ensure parent folders exist.
# - Save consistent UTF-8 text.
# ============================================================

def write_file(path, content):
    """
    Write text content to a file and create parent folders if needed.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content.strip() + "\n", encoding="utf-8")
    print(f"Created/updated: {path}")


# ============================================================
# 7. Helper function to export issue-specific ZIP files
# ============================================================
#
# Purpose:
# - Create one ZIP file per GitHub issue.
# - Each ZIP contains only the files needed for that issue.
# - You download these ZIPs from Colab and commit them locally later.
# ============================================================

def make_issue_zip(zip_name, files):
    """
    Create a ZIP file containing only the files needed for one GitHub issue.
    """
    export_dir = Path("exports")
    export_dir.mkdir(parents=True, exist_ok=True)

    zip_path = export_dir / zip_name

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for file_path in files:
            file_path = Path(file_path)

            if file_path.exists():
                zipf.write(file_path, arcname=str(file_path))
            else:
                print(f"Skipped missing file: {file_path}")

    print(f"Created ZIP: {zip_path}")
    return zip_path


# ============================================================
# 8. Helper function to run commands and show full errors
# ============================================================
#
# Purpose:
# - Run subprocess commands.
# - Print stdout and stderr clearly.
# - Fail loudly with useful error messages.
# ============================================================

def run_and_print(command, shell=False):
    """
    Run a command and print full stdout/stderr if it fails.
    """
    result = subprocess.run(
        command,
        shell=shell,
        capture_output=True,
        text=True,
    )

    print("\nCOMMAND:")
    print(command)

    print("\nSTDOUT:")
    print(result.stdout)

    print("\nSTDERR:")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")

    return result

Workspace created at: /content/ibrahim_mlops_project

Upload these files when prompted:
- train.csv
- test.csv
- WA_Fn-UseC_-Telco-Customer-Churn.csv.dvc
- prepare.py
- preprocess.py
- dvc.yaml
- params.yaml


Saving dvc.yaml to dvc.yaml
Saving params.yaml to params.yaml
Saving prepare.py to prepare.py
Saving preprocess.py to preprocess.py
Saving test.csv to test.csv
Saving train.csv to train.csv
Saving WA_Fn-UseC_-Telco-Customer-Churn.csv.dvc to WA_Fn-UseC_-Telco-Customer-Churn.csv.dvc
Uploaded: dvc.yaml
Uploaded: params.yaml
Uploaded: prepare.py
Uploaded: preprocess.py
Uploaded: test.csv
Uploaded: train.csv
Uploaded: WA_Fn-UseC_-Telco-Customer-Churn.csv.dvc
dvc.yaml is already in the project root. Skipping copy.

Files placed successfully:
- data/splits/train.csv
- data/splits/test.csv
- data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv.dvc
- src/data/prepare.py
- src/data/preprocess.py
- dvc.yaml
- configs/params.yaml

Dataset inspection:
Train shape: (400, 9)
Test shape: (100, 9)
Detected target column: target
Columns: ['feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'target']

Train target distribution:
target
0    202
1    198
Name: c

### **Issue #19 - Setup MLflow**

In [13]:
# ============================================================
# Issue #19 — Setup MLflow
# ============================================================
#
# Description:
# Configure MLflow tracking server.
#
# Tasks:
# - Configure local SQLite tracking backend.
# - Create reusable MLflow setup utility.
# - Verify tracking URI.
# - Start local MLflow UI in the background.
#
# Files:
# - src/__init__.py
# - src/training/__init__.py
# - src/training/mlflow_setup.py
# ============================================================

mlflow_setup_code = r'''
# =========================
# MLflow setup utilities
# =========================

from pathlib import Path

import mlflow


def configure_mlflow(
    experiment_name="mlops_training_experiments",
    backend_store_path="mlruns/mlflow.db",
):
    """
    Configure MLflow tracking using a local SQLite backend.

    SQLite is used instead of a plain file backend because it works better
    with experiment tracking and model registry workflows.
    """
    backend_store_path = Path(backend_store_path)
    backend_store_path.parent.mkdir(parents=True, exist_ok=True)

    tracking_uri = f"sqlite:///{backend_store_path.resolve()}"

    mlflow.set_tracking_uri(tracking_uri)
    mlflow.set_experiment(experiment_name)

    return tracking_uri
'''

write_file("src/training/mlflow_setup.py", mlflow_setup_code)

from src.training.mlflow_setup import configure_mlflow

tracking_uri = configure_mlflow()
print("\nIssue #19 verification:")
print("MLflow tracking URI:", tracking_uri)

subprocess.Popen(
    [
        "mlflow",
        "ui",
        "--backend-store-uri",
        "sqlite:///mlruns/mlflow.db",
        "--host",
        "0.0.0.0",
        "--port",
        "5000",
    ],
    stdout=open("mlflow.log", "w"),
    stderr=subprocess.STDOUT,
)

print("MLflow UI server started in background on port 5000.")

make_issue_zip(
    "issue_19_setup_mlflow.zip",
    [
        "src/__init__.py",
        "src/training/__init__.py",
        "src/training/mlflow_setup.py",
    ],
)

Created/updated: src/training/mlflow_setup.py

Issue #19 verification:
MLflow tracking URI: sqlite:////content/ibrahim_mlops_project/mlruns/mlflow.db
MLflow UI server started in background on port 5000.
Created ZIP: exports/issue_19_setup_mlflow.zip


PosixPath('exports/issue_19_setup_mlflow.zip')

### **Issue #18 - Model Evaluation**

In [14]:
# ============================================================
# Issue #18 — Model Evaluation
# ============================================================
#
# Description:
# Create reusable model evaluation and diagnostics utilities.
#
# Tasks:
# - Detect classification vs regression.
# - Compute classification metrics.
# - Compute regression metrics.
# - Save metrics.json.
# - Save classification report.
# - Save confusion matrix.
# - Add overfitting / underfitting diagnostics.
# - Add data leakage checks.
# - Remove data drift checks completely.
#
# Files:
# - src/evaluation/__init__.py
# - src/evaluation/evaluate.py
# - src/evaluation/diagnostics.py
# ============================================================

evaluate_code = r'''
# =========================
# Model evaluation utilities
# =========================

import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)


def detect_problem_type(y):
    """
    Detect whether the target represents classification or regression.

    Classification is assumed when:
    - target dtype is object/category/bool, or
    - the number of unique values is reasonably small.
    """
    y_series = pd.Series(y)
    unique_count = y_series.nunique()

    if str(y_series.dtype) in ["object", "category", "bool"] or unique_count <= 20:
        return "classification"

    return "regression"


def evaluate_model(model, X_test, y_test, output_dir="reports"):
    """
    Evaluate a trained model and save metrics/reports.

    For classification:
    - accuracy
    - macro precision
    - macro recall
    - macro F1
    - ROC-AUC when probability predictions are available
    - classification report
    - confusion matrix

    For regression:
    - MAE
    - MSE
    - RMSE
    - R2
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    problem_type = detect_problem_type(y_test)
    y_pred = model.predict(X_test)

    metrics = {
        "problem_type": problem_type,
    }

    if problem_type == "classification":
        metrics["accuracy"] = float(accuracy_score(y_test, y_pred))
        metrics["precision_macro"] = float(
            precision_score(y_test, y_pred, average="macro", zero_division=0)
        )
        metrics["recall_macro"] = float(
            recall_score(y_test, y_pred, average="macro", zero_division=0)
        )
        metrics["f1_macro"] = float(
            f1_score(y_test, y_pred, average="macro", zero_division=0)
        )

        if hasattr(model, "predict_proba"):
            try:
                y_proba = model.predict_proba(X_test)

                if y_proba.shape[1] == 2:
                    metrics["roc_auc"] = float(roc_auc_score(y_test, y_proba[:, 1]))

                elif y_proba.shape[1] > 2:
                    metrics["roc_auc_ovr"] = float(
                        roc_auc_score(y_test, y_proba, multi_class="ovr")
                    )

            except Exception:
                pass

        report = classification_report(y_test, y_pred, zero_division=0)
        confusion = confusion_matrix(y_test, y_pred)

        with open(output_path / "classification_report.txt", "w", encoding="utf-8") as file:
            file.write(report)

        pd.DataFrame(confusion).to_csv(
            output_path / "confusion_matrix.csv",
            index=False,
        )

    else:
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)

        metrics["mae"] = float(mean_absolute_error(y_test, y_pred))
        metrics["mse"] = float(mse)
        metrics["rmse"] = float(rmse)
        metrics["r2"] = float(r2_score(y_test, y_pred))

    with open(output_path / "metrics.json", "w", encoding="utf-8") as file:
        json.dump(metrics, file, indent=4)

    return metrics
'''

diagnostics_code = r'''
# =========================
# Model and leakage diagnostics
# =========================

import json
from pathlib import Path

import pandas as pd


def _safe_float(value):
    """
    Convert values to normal Python floats for JSON serialization.
    """
    try:
        if pd.isna(value):
            return None
        return float(value)
    except Exception:
        return None


def check_data_leakage(
    train_df,
    test_df,
    target_column,
    output_dir="reports",
    high_corr_threshold=0.98,
):
    """
    Run practical data leakage checks.

    Checks included:
    1. Duplicate feature rows between train and test.
    2. Feature columns that are identical to the target.
    3. Numeric features that are suspiciously correlated with the target.
    4. Categorical features that almost perfectly identify the target.

    These checks cannot prove there is no leakage, but they catch common problems.
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    warnings = []
    details = {}

    if target_column not in train_df.columns:
        raise ValueError(f"Target column '{target_column}' not found in train data.")

    if target_column not in test_df.columns:
        raise ValueError(f"Target column '{target_column}' not found in test data.")

    feature_columns = [col for col in train_df.columns if col != target_column]

    train_feature_hashes = pd.util.hash_pandas_object(
        train_df[feature_columns].astype(str),
        index=False,
    )

    test_feature_hashes = pd.util.hash_pandas_object(
        test_df[feature_columns].astype(str),
        index=False,
    )

    overlap_count = int(len(set(train_feature_hashes).intersection(set(test_feature_hashes))))
    details["train_test_feature_overlap_count"] = overlap_count

    if overlap_count > 0:
        warnings.append(
            f"Possible leakage: {overlap_count} duplicated feature rows found across train and test."
        )

    identical_to_target = []

    for col in feature_columns:
        try:
            if train_df[col].reset_index(drop=True).equals(
                train_df[target_column].reset_index(drop=True)
            ):
                identical_to_target.append(col)
        except Exception:
            pass

    details["features_identical_to_target"] = identical_to_target

    if identical_to_target:
        warnings.append(
            f"High leakage risk: features identical to target found: {identical_to_target}"
        )

    high_target_correlations = {}

    numeric_columns = train_df[feature_columns].select_dtypes(
        include=["int64", "float64", "int32", "float32"]
    ).columns.tolist()

    if pd.api.types.is_numeric_dtype(train_df[target_column]):
        for col in numeric_columns:
            try:
                corr = train_df[col].corr(train_df[target_column])
                corr_abs = abs(corr)

                if corr_abs >= high_corr_threshold:
                    high_target_correlations[col] = _safe_float(corr)

            except Exception:
                pass

    details["high_target_correlations"] = high_target_correlations

    if high_target_correlations:
        warnings.append(
            f"Possible leakage: very high feature-target correlations found: {high_target_correlations}"
        )

    suspicious_categorical_features = {}

    categorical_columns = train_df[feature_columns].select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    for col in categorical_columns:
        try:
            crosstab = pd.crosstab(train_df[col], train_df[target_column], normalize="index")

            if not crosstab.empty:
                max_class_purity = crosstab.max(axis=1).max()

                if max_class_purity >= high_corr_threshold:
                    suspicious_categorical_features[col] = _safe_float(max_class_purity)

        except Exception:
            pass

    details["suspicious_categorical_features"] = suspicious_categorical_features

    if suspicious_categorical_features:
        warnings.append(
            "Possible leakage: categorical features almost perfectly identify the target: "
            f"{suspicious_categorical_features}"
        )

    status = "PASS" if not warnings else "WARNING"

    report = {
        "status": status,
        "warnings": warnings,
        "details": details,
    }

    report_path = output_path / "data_leakage_report.json"

    with open(report_path, "w", encoding="utf-8") as file:
        json.dump(report, file, indent=4)

    return report


def evaluate_overfit_underfit(
    model_name,
    train_metrics,
    test_metrics,
    problem_type,
    primary_metric,
    higher_is_better=True,
    overfit_gap_threshold=0.10,
    classification_good_threshold=0.75,
    classification_underfit_threshold=0.60,
):
    """
    Create a model quality verdict based on train/test performance.

    Classification:
    - Overfitting: train score is much higher than test score.
    - Underfitting: train and test scores are both weak.
    - Weak model: test score is below a reasonable threshold.

    Regression:
    - Uses lower-is-better metric such as RMSE.
    """
    warnings = []
    verdict = "GOOD"

    train_score = train_metrics.get(primary_metric)
    test_score = test_metrics.get(primary_metric)

    if train_score is None or test_score is None:
        return {
            "model_name": model_name,
            "verdict": "UNKNOWN",
            "warnings": [f"Could not evaluate overfitting because '{primary_metric}' is missing."],
            "primary_metric": primary_metric,
            "train_score": train_score,
            "test_score": test_score,
            "gap": None,
        }

    if higher_is_better:
        gap = train_score - test_score

        if gap > overfit_gap_threshold:
            verdict = "OVERFITTING_WARNING"
            warnings.append(
                f"Possible overfitting: train {primary_metric}={train_score:.4f}, "
                f"test {primary_metric}={test_score:.4f}, gap={gap:.4f}."
            )

        elif train_score < classification_underfit_threshold and test_score < classification_underfit_threshold:
            verdict = "UNDERFITTING_WARNING"
            warnings.append(
                f"Possible underfitting: both train and test {primary_metric} are low "
                f"(train={train_score:.4f}, test={test_score:.4f})."
            )

        elif test_score < classification_good_threshold:
            verdict = "WEAK_MODEL_WARNING"
            warnings.append(
                f"Model may not be strong enough: test {primary_metric}={test_score:.4f} "
                f"is below {classification_good_threshold:.2f}."
            )

        else:
            warnings.append(
                f"Model looks acceptable: test {primary_metric}={test_score:.4f}, "
                f"gap={gap:.4f}."
            )

    else:
        gap = test_score - train_score

        if gap > overfit_gap_threshold:
            verdict = "OVERFITTING_WARNING"
            warnings.append(
                f"Possible overfitting: train {primary_metric}={train_score:.4f}, "
                f"test {primary_metric}={test_score:.4f}, test is worse by {gap:.4f}."
            )

        else:
            warnings.append(
                f"Model looks acceptable based on {primary_metric}: "
                f"train={train_score:.4f}, test={test_score:.4f}."
            )

    return {
        "model_name": model_name,
        "verdict": verdict,
        "warnings": warnings,
        "primary_metric": primary_metric,
        "train_score": _safe_float(train_score),
        "test_score": _safe_float(test_score),
        "gap": _safe_float(gap),
    }


def save_model_diagnostics_report(
    diagnostics,
    output_dir="reports",
    filename="model_diagnostics_report.json",
):
    """
    Save model diagnostics to JSON.
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    report_path = output_path / filename

    with open(report_path, "w", encoding="utf-8") as file:
        json.dump(diagnostics, file, indent=4)

    return report_path
'''

write_file("src/evaluation/evaluate.py", evaluate_code)
write_file("src/evaluation/diagnostics.py", diagnostics_code)

make_issue_zip(
    "issue_18_model_evaluation.zip",
    [
        "src/__init__.py",
        "src/evaluation/__init__.py",
        "src/evaluation/evaluate.py",
        "src/evaluation/diagnostics.py",
    ],
)

Created/updated: src/evaluation/evaluate.py
Created/updated: src/evaluation/diagnostics.py
Created ZIP: exports/issue_18_model_evaluation.zip


PosixPath('exports/issue_18_model_evaluation.zip')

### **Issue #17 - Train Baseline Model**

In [15]:
# ============================================================
# Issue #17 — Train Baseline Model
# ============================================================
#
# Description:
# Train an initial baseline model.
#
# Tasks:
# - Load train/test split files.
# - Build leakage-safe preprocessing pipeline.
# - Train baseline Random Forest model.
# - Save model artifact.
# - Save baseline metadata.
#
# Files:
# - src/training/train.py
#
# Important:
# - This is the baseline-only version of train.py.
# - Issue #20 later replaces train.py with the MLflow experiment version.
# ============================================================

baseline_train_code = r'''
# =========================
# Baseline model training
# =========================

import argparse
import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.evaluation.evaluate import detect_problem_type, evaluate_model


def make_one_hot_encoder():
    """
    Create OneHotEncoder with compatibility across sklearn versions.
    """
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def load_split_data(train_path, test_path, target_column):
    """
    Load train/test split files and separate features from target.
    """
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    if target_column not in train_df.columns:
        raise ValueError(f"Target column '{target_column}' was not found in train data.")

    if target_column not in test_df.columns:
        raise ValueError(f"Target column '{target_column}' was not found in test data.")

    X_train = train_df.drop(columns=[target_column])
    y_train = train_df[target_column]

    X_test = test_df.drop(columns=[target_column])
    y_test = test_df[target_column]

    return X_train, y_train, X_test, y_test


def build_preprocessor(X_train):
    """
    Build preprocessing for numeric and categorical features.

    Numeric features:
    - median imputation
    - standard scaling

    Categorical features:
    - most frequent imputation
    - one-hot encoding
    """
    numeric_features = X_train.select_dtypes(
        include=["int64", "float64", "int32", "float32"]
    ).columns.tolist()

    categorical_features = X_train.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", make_one_hot_encoder()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_features),
            ("categorical", categorical_pipeline, categorical_features),
        ]
    )

    return preprocessor, numeric_features, categorical_features


def build_baseline_model(problem_type, random_state):
    """
    Build baseline model based on problem type.
    """
    if problem_type == "classification":
        return RandomForestClassifier(
            n_estimators=200,
            random_state=random_state,
            n_jobs=-1,
        )

    return RandomForestRegressor(
        n_estimators=200,
        random_state=random_state,
        n_jobs=-1,
    )


def train_baseline(args):
    """
    Train baseline model and save artifact.
    """
    Path(args.model_dir).mkdir(parents=True, exist_ok=True)
    Path(args.report_dir).mkdir(parents=True, exist_ok=True)

    X_train, y_train, X_test, y_test = load_split_data(
        train_path=args.train_path,
        test_path=args.test_path,
        target_column=args.target_column,
    )

    problem_type = detect_problem_type(y_train)

    preprocessor, numeric_features, categorical_features = build_preprocessor(X_train)
    model = build_baseline_model(problem_type, args.random_state)

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    pipeline.fit(X_train, y_train)

    metrics = evaluate_model(
        model=pipeline,
        X_test=X_test,
        y_test=y_test,
        output_dir=args.report_dir,
    )

    model_path = Path(args.model_dir) / "baseline_model.pkl"
    joblib.dump(pipeline, model_path)

    metadata = {
        "target_column": args.target_column,
        "problem_type": problem_type,
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "input_columns": X_train.columns.tolist(),
        "model_path": str(model_path),
        "metrics": metrics,
    }

    metadata_path = Path(args.model_dir) / "baseline_metadata.json"

    with open(metadata_path, "w", encoding="utf-8") as file:
        json.dump(metadata, file, indent=4)

    print("Baseline training completed.")
    print(f"Model saved to: {model_path}")
    print(f"Metadata saved to: {metadata_path}")
    print(metrics)


def parse_args():
    """
    Parse command line arguments.
    """
    parser = argparse.ArgumentParser()

    parser.add_argument("--train-path", default="data/splits/train.csv")
    parser.add_argument("--test-path", default="data/splits/test.csv")
    parser.add_argument("--target-column", default="target")
    parser.add_argument("--model-dir", default="models")
    parser.add_argument("--report-dir", default="reports")
    parser.add_argument("--random-state", type=int, default=42)
    parser.add_argument("--registry-summary-path", default="reports/model_registry_summary.json",)
    return parser.parse_args()


if __name__ == "__main__":
    train_baseline(parse_args())
'''

write_file("src/training/train.py", baseline_train_code)

print("\nRunning baseline training for Issue #17...")
run_and_print(
    [
        sys.executable,
        "src/training/train.py",
        "--train-path",
        "data/splits/train.csv",
        "--test-path",
        "data/splits/test.csv",
        "--target-column",
        TARGET_COLUMN,
    ]
)

make_issue_zip(
    "issue_17_train_baseline.zip",
    [
        "src/training/train.py",
    ],
)

Created/updated: src/training/train.py

Running baseline training for Issue #17...

COMMAND:
['/usr/bin/python3', 'src/training/train.py', '--train-path', 'data/splits/train.csv', '--test-path', 'data/splits/test.csv', '--target-column', 'target']

STDOUT:
Baseline training completed.
Model saved to: models/baseline_model.pkl
Metadata saved to: models/baseline_metadata.json
{'problem_type': 'classification', 'accuracy': 0.94, 'precision_macro': 0.94, 'recall_macro': 0.94, 'f1_macro': 0.94, 'roc_auc': 0.9798}


STDERR:

Created ZIP: exports/issue_17_train_baseline.zip


PosixPath('exports/issue_17_train_baseline.zip')

### **Issue #20 - Log Experiments**

In [16]:
# ============================================================
# Issue #20 — Log Experiments
# ============================================================
#
# Description:
# Train multiple models with MLflow tracking and compare:
# - All-feature models
# - Feature-importance-selected models
#
# Main rule:
# - Train models using all features first.
# - Record the all-feature best model and metrics.
# - Run permutation feature importance as analysis.
# - Test multiple selected-feature cutoffs.
# - If selected features improve performance, use selected-feature model.
# - If selected features do not improve performance, keep all-feature best model as final.
#
# Tasks:
# - Log parameters, metrics, and artifacts to MLflow.
# - Train Logistic Regression, Random Forest, Gradient Boosting, SVC.
# - Train XGBoost, LightGBM, CatBoost if available.
# - Run data leakage checks.
# - Run overfitting / underfitting diagnostics.
# - Run permutation feature importance.
# - Test top 5, top 10, top 15, top 20, cumulative 90%, cumulative 95%.
# - Save final best model.
# - Save feature importance analysis even if not final.
#
# Files:
# - src/training/train.py
# ============================================================

mlflow_train_code = r'''
# =========================
# Training with MLflow experiment logging, diagnostics, and feature-importance analysis
# =========================

import argparse
import importlib
import json
from pathlib import Path

import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

from src.evaluation.diagnostics import (
    check_data_leakage,
    evaluate_overfit_underfit,
    save_model_diagnostics_report,
)
from src.evaluation.evaluate import detect_problem_type, evaluate_model
from src.training.mlflow_setup import configure_mlflow


def optional_import(module_name, class_name):
    """
    Import optional model classes safely.
    """
    try:
        module = importlib.import_module(module_name)
        return getattr(module, class_name)
    except Exception as error:
        print(f"Optional model skipped: {class_name} from {module_name}. Reason: {error}")
        return None


def make_one_hot_encoder():
    """
    Create OneHotEncoder with compatibility across sklearn versions.
    """
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def load_split_data(train_path, test_path, target_column):
    """
    Load train/test split files and separate features from target.
    """
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    if target_column not in train_df.columns:
        raise ValueError(f"Target column '{target_column}' was not found in train data.")

    if target_column not in test_df.columns:
        raise ValueError(f"Target column '{target_column}' was not found in test data.")

    X_train = train_df.drop(columns=[target_column])
    y_train = train_df[target_column]

    X_test = test_df.drop(columns=[target_column])
    y_test = test_df[target_column]

    return train_df, test_df, X_train, y_train, X_test, y_test


def build_preprocessor(X_train):
    """
    Build preprocessing for numeric and categorical features.
    """
    numeric_features = X_train.select_dtypes(
        include=["int64", "float64", "int32", "float32"]
    ).columns.tolist()

    categorical_features = X_train.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", make_one_hot_encoder()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_features),
            ("categorical", categorical_pipeline, categorical_features),
        ]
    )

    return preprocessor, numeric_features, categorical_features


def get_candidate_models(problem_type, random_state):
    """
    Return candidate models for MLflow experiment comparison.
    """
    models = {}

    if problem_type == "classification":
        models["logistic_regression"] = LogisticRegression(
            max_iter=1000,
            random_state=random_state,
        )

        models["random_forest"] = RandomForestClassifier(
            n_estimators=200,
            random_state=random_state,
            n_jobs=-1,
        )

        models["gradient_boosting"] = GradientBoostingClassifier(
            random_state=random_state,
        )

        models["svc"] = SVC(
            probability=True,
            random_state=random_state,
        )

        XGBClassifier = optional_import("xgboost", "XGBClassifier")
        LGBMClassifier = optional_import("lightgbm", "LGBMClassifier")
        CatBoostClassifier = optional_import("catboost", "CatBoostClassifier")

        if XGBClassifier is not None:
            models["xgboost"] = XGBClassifier(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=4,
                subsample=0.9,
                colsample_bytree=0.9,
                eval_metric="logloss",
                random_state=random_state,
            )

        if LGBMClassifier is not None:
            models["lightgbm"] = LGBMClassifier(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=-1,
                random_state=random_state,
                verbosity=-1,
            )

        if CatBoostClassifier is not None:
            models["catboost"] = CatBoostClassifier(
                iterations=200,
                learning_rate=0.05,
                depth=4,
                random_seed=random_state,
                verbose=False,
            )

    else:
        models["ridge_regression"] = Ridge()

        models["random_forest"] = RandomForestRegressor(
            n_estimators=200,
            random_state=random_state,
            n_jobs=-1,
        )

        models["gradient_boosting"] = GradientBoostingRegressor(
            random_state=random_state,
        )

        XGBRegressor = optional_import("xgboost", "XGBRegressor")
        LGBMRegressor = optional_import("lightgbm", "LGBMRegressor")
        CatBoostRegressor = optional_import("catboost", "CatBoostRegressor")

        if XGBRegressor is not None:
            models["xgboost"] = XGBRegressor(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=4,
                subsample=0.9,
                colsample_bytree=0.9,
                random_state=random_state,
            )

        if LGBMRegressor is not None:
            models["lightgbm"] = LGBMRegressor(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=-1,
                random_state=random_state,
                verbosity=-1,
            )

        if CatBoostRegressor is not None:
            models["catboost"] = CatBoostRegressor(
                iterations=200,
                learning_rate=0.05,
                depth=4,
                random_seed=random_state,
                verbose=False,
            )

    return models


def get_primary_metric(problem_type):
    """
    Select the primary metric used for model comparison.
    """
    if problem_type == "classification":
        return "f1_macro", True

    return "rmse", False


def get_scorer_name(problem_type):
    """
    Return sklearn scoring name for permutation importance.
    """
    if problem_type == "classification":
        return "f1_macro"

    return "neg_root_mean_squared_error"


def make_train_validation_split(X_train, y_train, problem_type, validation_size, random_state):
    """
    Create train-core and validation split for feature-importance analysis only.
    """
    stratify_values = None

    if problem_type == "classification":
        class_counts = pd.Series(y_train).value_counts()

        if class_counts.min() >= 2:
            stratify_values = y_train

    return train_test_split(
        X_train,
        y_train,
        test_size=validation_size,
        random_state=random_state,
        stratify=stratify_values,
    )


def compute_permutation_feature_importance(
    pipeline,
    X_valid,
    y_valid,
    problem_type,
    output_dir,
    random_state,
    n_repeats,
):
    """
    Compute permutation importance on original input features.

    This is more reliable than basic tree split importance because it measures
    the validation performance drop when each feature is shuffled.
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    scoring = get_scorer_name(problem_type)

    result = permutation_importance(
        estimator=pipeline,
        X=X_valid,
        y=y_valid,
        scoring=scoring,
        n_repeats=n_repeats,
        random_state=random_state,
        n_jobs=-1,
    )

    importance_df = pd.DataFrame(
        {
            "feature": X_valid.columns.tolist(),
            "importance_mean": result.importances_mean,
            "importance_std": result.importances_std,
        }
    )

    importance_df["importance_positive"] = importance_df["importance_mean"].clip(lower=0)
    importance_df = importance_df.sort_values("importance_mean", ascending=False)

    total_positive_importance = importance_df["importance_positive"].sum()

    if total_positive_importance > 0:
        importance_df["importance_ratio"] = (
            importance_df["importance_positive"] / total_positive_importance
        )
        importance_df["cumulative_importance"] = importance_df["importance_ratio"].cumsum()
    else:
        importance_df["importance_ratio"] = 0.0
        importance_df["cumulative_importance"] = 0.0

    importance_df.to_csv(output_path / "all_feature_importance.csv", index=False)
    importance_df.head(10).to_csv(output_path / "top_10_feature_importance.csv", index=False)

    top_10_payload = {
        "method": "permutation_importance",
        "scoring": scoring,
        "top_10_features": importance_df.head(10)["feature"].tolist(),
    }

    with open(output_path / "top_10_features.json", "w", encoding="utf-8") as file:
        json.dump(top_10_payload, file, indent=4)

    return importance_df, top_10_payload


def build_feature_cutoff_candidates(importance_df):
    """
    Build multiple selected-feature candidates instead of blindly using top 10.
    """
    ranked_features = importance_df["feature"].tolist()
    feature_count = len(ranked_features)

    candidates = {}

    for cutoff in [5, 10, 15, 20]:
        safe_cutoff = min(cutoff, feature_count)

        if safe_cutoff > 0:
            candidates[f"top_{safe_cutoff}"] = ranked_features[:safe_cutoff]

    if "cumulative_importance" in importance_df.columns:
        for threshold in [0.90, 0.95]:
            covered = importance_df[
                importance_df["cumulative_importance"] <= threshold
            ]["feature"].tolist()

            if len(covered) == 0 and feature_count > 0:
                covered = [ranked_features[0]]

            if len(covered) < feature_count:
                next_index = len(covered)

                if next_index < feature_count:
                    covered = ranked_features[: next_index + 1]

            candidates[f"cumulative_{int(threshold * 100)}"] = covered

    unique_candidates = {}
    seen = set()

    for candidate_name, features in candidates.items():
        feature_tuple = tuple(features)

        if feature_tuple not in seen:
            unique_candidates[candidate_name] = features
            seen.add(feature_tuple)

    return unique_candidates


def select_best_feature_subset(
    reference_estimator,
    X_train_core,
    y_train_core,
    X_valid,
    y_valid,
    feature_candidates,
    problem_type,
    primary_metric,
    higher_is_better,
):
    """
    Train the best all-feature model family using multiple feature subsets.
    """
    results = []
    best_result = None

    for candidate_name, selected_features in feature_candidates.items():
        preprocessor, numeric_features, categorical_features = build_preprocessor(
            X_train_core[selected_features]
        )

        pipeline = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("model", clone(reference_estimator)),
            ]
        )

        pipeline.fit(X_train_core[selected_features], y_train_core)

        train_metrics = evaluate_model(
            model=pipeline,
            X_test=X_train_core[selected_features],
            y_test=y_train_core,
            output_dir="reports",
        )

        validation_metrics = evaluate_model(
            model=pipeline,
            X_test=X_valid[selected_features],
            y_test=y_valid,
            output_dir="reports",
        )

        score = validation_metrics[primary_metric]

        result = {
            "candidate_name": candidate_name,
            "feature_count": len(selected_features),
            "selected_features": selected_features,
            "validation_score": float(score),
            "train_metrics": train_metrics,
            "validation_metrics": validation_metrics,
        }

        results.append(result)

        if best_result is None:
            best_result = result

        else:
            if higher_is_better:
                if score > best_result["validation_score"]:
                    best_result = result
                elif score == best_result["validation_score"] and len(selected_features) < best_result["feature_count"]:
                    best_result = result
            else:
                if score < best_result["validation_score"]:
                    best_result = result
                elif score == best_result["validation_score"] and len(selected_features) < best_result["feature_count"]:
                    best_result = result

    return best_result, results


def train_with_mlflow(args):
    """
    Train all-feature models, run feature-importance analysis, compare selected-feature
    performance against all-feature performance, and save the true final best model.
    """
    Path(args.model_dir).mkdir(parents=True, exist_ok=True)
    Path(args.report_dir).mkdir(parents=True, exist_ok=True)
    Path(args.feature_importance_dir).mkdir(parents=True, exist_ok=True)

    configure_mlflow(
        experiment_name=args.experiment_name,
        backend_store_path=args.backend_store_path,
    )

    train_df, test_df, X_train, y_train, X_test, y_test = load_split_data(
        train_path=args.train_path,
        test_path=args.test_path,
        target_column=args.target_column,
    )

    problem_type = detect_problem_type(y_train)
    primary_metric, higher_is_better = get_primary_metric(problem_type)

    leakage_report = check_data_leakage(
        train_df=train_df,
        test_df=test_df,
        target_column=args.target_column,
        output_dir=args.report_dir,
    )

    candidate_models = get_candidate_models(problem_type, args.random_state)

    best_all_feature_score = None
    best_all_feature_model_name = None
    best_all_feature_model = None
    best_all_feature_estimator = None
    best_all_feature_train_metrics = None
    best_all_feature_test_metrics = None
    best_all_feature_diagnostics = None

    run_summaries = []
    diagnostic_summaries = []

    for model_name, estimator in candidate_models.items():
        preprocessor, numeric_features, categorical_features = build_preprocessor(X_train)

        with mlflow.start_run(run_name=f"experiment_{model_name}_all_features"):
            pipeline = Pipeline(
                steps=[
                    ("preprocessor", preprocessor),
                    ("model", estimator),
                ]
            )

            pipeline.fit(X_train, y_train)

            train_metrics = evaluate_model(
                model=pipeline,
                X_test=X_train,
                y_test=y_train,
                output_dir=args.report_dir,
            )

            test_metrics = evaluate_model(
                model=pipeline,
                X_test=X_test,
                y_test=y_test,
                output_dir=args.report_dir,
            )

            diagnostics = evaluate_overfit_underfit(
                model_name=f"{model_name}_all_features",
                train_metrics=train_metrics,
                test_metrics=test_metrics,
                problem_type=problem_type,
                primary_metric=primary_metric,
                higher_is_better=higher_is_better,
            )

            diagnostic_summaries.append(diagnostics)

            score = test_metrics[primary_metric]

            mlflow.log_param("model_name", model_name)
            mlflow.log_param("model_stage", "all_features_reference")
            mlflow.log_param("problem_type", problem_type)
            mlflow.log_param("target_column", args.target_column)
            mlflow.log_param("train_rows", X_train.shape[0])
            mlflow.log_param("test_rows", X_test.shape[0])
            mlflow.log_param("numeric_features", len(numeric_features))
            mlflow.log_param("categorical_features", len(categorical_features))
            mlflow.log_param("primary_metric", primary_metric)
            mlflow.log_param("quality_verdict", diagnostics["verdict"])
            mlflow.log_param("leakage_status", leakage_report["status"])
            mlflow.log_param("feature_selection_used", "false")
            mlflow.log_param("is_final_model", "false")

            for metric_name, metric_value in train_metrics.items():
                if isinstance(metric_value, (int, float)):
                    mlflow.log_metric(f"train_{metric_name}", metric_value)

            for metric_name, metric_value in test_metrics.items():
                if isinstance(metric_value, (int, float)):
                    mlflow.log_metric(f"test_{metric_name}", metric_value)
                    mlflow.log_metric(metric_name, metric_value)

            if diagnostics["gap"] is not None:
                mlflow.log_metric("train_test_gap", diagnostics["gap"])

            for artifact_path in [
                Path(args.report_dir) / "metrics.json",
                Path(args.report_dir) / "classification_report.txt",
                Path(args.report_dir) / "confusion_matrix.csv",
                Path(args.report_dir) / "data_leakage_report.json",
            ]:
                if artifact_path.exists():
                    mlflow.log_artifact(str(artifact_path))

            mlflow.sklearn.log_model(
                sk_model=pipeline,
                artifact_path="model",
            )

            run_summary = {
                "model_name": model_name,
                "model_stage": "all_features_reference",
                "primary_metric": primary_metric,
                "score": score,
                "test_score": score,
                "feature_selection_used": False,
                "quality_verdict": diagnostics["verdict"],
                "diagnostic_warning": " | ".join(diagnostics["warnings"]),
                **{f"train_{k}": v for k, v in train_metrics.items()},
                **{f"test_{k}": v for k, v in test_metrics.items()},
            }

            run_summaries.append(run_summary)

            should_replace = (
                best_all_feature_score is None
                or (higher_is_better and score > best_all_feature_score)
                or (not higher_is_better and score < best_all_feature_score)
            )

            if should_replace:
                best_all_feature_score = score
                best_all_feature_model_name = model_name
                best_all_feature_model = pipeline
                best_all_feature_estimator = estimator
                best_all_feature_train_metrics = train_metrics
                best_all_feature_test_metrics = test_metrics
                best_all_feature_diagnostics = diagnostics

    best_all_features_model_path = Path(args.model_dir) / "best_all_features_model.pkl"
    best_reference_model_path = Path(args.model_dir) / "best_reference_model.pkl"

    joblib.dump(best_all_feature_model, best_all_features_model_path)
    joblib.dump(best_all_feature_model, best_reference_model_path)

    X_train_core, X_valid, y_train_core, y_valid = make_train_validation_split(
        X_train=X_train,
        y_train=y_train,
        problem_type=problem_type,
        validation_size=args.validation_size,
        random_state=args.random_state,
    )

    reference_preprocessor, _, _ = build_preprocessor(X_train_core)

    reference_pipeline_for_importance = Pipeline(
        steps=[
            ("preprocessor", reference_preprocessor),
            ("model", clone(best_all_feature_estimator)),
        ]
    )

    reference_pipeline_for_importance.fit(X_train_core, y_train_core)

    importance_df, top_10_payload = compute_permutation_feature_importance(
        pipeline=reference_pipeline_for_importance,
        X_valid=X_valid,
        y_valid=y_valid,
        problem_type=problem_type,
        output_dir=args.feature_importance_dir,
        random_state=args.random_state,
        n_repeats=args.permutation_repeats,
    )

    feature_candidates = build_feature_cutoff_candidates(importance_df)

    selected_feature_result, feature_selection_results = select_best_feature_subset(
        reference_estimator=best_all_feature_estimator,
        X_train_core=X_train_core,
        y_train_core=y_train_core,
        X_valid=X_valid,
        y_valid=y_valid,
        feature_candidates=feature_candidates,
        problem_type=problem_type,
        primary_metric=primary_metric,
        higher_is_better=higher_is_better,
    )

    selected_features = selected_feature_result["selected_features"]

    feature_selection_results_path = Path(args.feature_importance_dir) / "feature_selection_results.json"

    with open(feature_selection_results_path, "w", encoding="utf-8") as file:
        json.dump(feature_selection_results, file, indent=4)

    selected_features_payload = {
        "method": "permutation_importance_with_validation_cutoff_selection",
        "reference_best_model_name": best_all_feature_model_name,
        "primary_metric": primary_metric,
        "selected_candidate_name": selected_feature_result["candidate_name"],
        "selected_feature_count": len(selected_features),
        "selected_features": selected_features,
        "top_10_reference_features": top_10_payload["top_10_features"],
        "feature_selection_validation_score": selected_feature_result["validation_score"],
    }

    selected_features_path = Path(args.feature_importance_dir) / "selected_features.json"

    with open(selected_features_path, "w", encoding="utf-8") as file:
        json.dump(selected_features_payload, file, indent=4)

    selected_preprocessor, selected_numeric_features, selected_categorical_features = build_preprocessor(
        X_train[selected_features]
    )

    selected_feature_pipeline = Pipeline(
        steps=[
            ("preprocessor", selected_preprocessor),
            ("model", clone(best_all_feature_estimator)),
        ]
    )

    selected_feature_pipeline.fit(X_train[selected_features], y_train)

    selected_train_metrics = evaluate_model(
        model=selected_feature_pipeline,
        X_test=X_train[selected_features],
        y_test=y_train,
        output_dir=args.report_dir,
    )

    selected_test_metrics = evaluate_model(
        model=selected_feature_pipeline,
        X_test=X_test[selected_features],
        y_test=y_test,
        output_dir=args.report_dir,
    )

    selected_diagnostics = evaluate_overfit_underfit(
        model_name=f"{best_all_feature_model_name}_selected_features",
        train_metrics=selected_train_metrics,
        test_metrics=selected_test_metrics,
        problem_type=problem_type,
        primary_metric=primary_metric,
        higher_is_better=higher_is_better,
    )

    diagnostic_summaries.append(selected_diagnostics)

    selected_score = selected_test_metrics[primary_metric]

    with mlflow.start_run(run_name=f"analysis_{best_all_feature_model_name}_selected_features"):
        mlflow.log_param("model_name", f"{best_all_feature_model_name}_selected_features")
        mlflow.log_param("model_stage", "selected_features_analysis")
        mlflow.log_param("reference_best_model_name", best_all_feature_model_name)
        mlflow.log_param("problem_type", problem_type)
        mlflow.log_param("target_column", args.target_column)
        mlflow.log_param("primary_metric", primary_metric)
        mlflow.log_param("feature_importance_method", "permutation_importance")
        mlflow.log_param("feature_selection_method", "validation_cutoff_selection")
        mlflow.log_param("selected_candidate_name", selected_feature_result["candidate_name"])
        mlflow.log_param("selected_feature_count", len(selected_features))
        mlflow.log_param("selected_features", json.dumps(selected_features))
        mlflow.log_param("top_10_reference_features", json.dumps(top_10_payload["top_10_features"]))
        mlflow.log_param("quality_verdict", selected_diagnostics["verdict"])
        mlflow.log_param("leakage_status", leakage_report["status"])
        mlflow.log_param("feature_selection_used", "true")
        mlflow.log_param("is_final_model", "false")

        for metric_name, metric_value in selected_train_metrics.items():
            if isinstance(metric_value, (int, float)):
                mlflow.log_metric(f"selected_train_{metric_name}", metric_value)

        for metric_name, metric_value in selected_test_metrics.items():
            if isinstance(metric_value, (int, float)):
                mlflow.log_metric(f"selected_test_{metric_name}", metric_value)
                mlflow.log_metric(metric_name, metric_value)

        if selected_diagnostics["gap"] is not None:
            mlflow.log_metric("selected_train_test_gap", selected_diagnostics["gap"])

        selected_summary = {
            "reference_best_model_name": best_all_feature_model_name,
            "selected_model_name": f"{best_all_feature_model_name}_selected_features",
            "primary_metric": primary_metric,
            "all_features_score": float(best_all_feature_score),
            "selected_features_score": float(selected_score),
            "selected_features": selected_features,
            "selected_feature_count": len(selected_features),
            "selected_candidate_name": selected_feature_result["candidate_name"],
            "selected_train_metrics": selected_train_metrics,
            "selected_test_metrics": selected_test_metrics,
            "selected_diagnostics": selected_diagnostics,
            "feature_importance_kept_as_analysis": True,
        }

        selected_summary_path = Path(args.report_dir) / "selected_feature_model_summary.json"

        with open(selected_summary_path, "w", encoding="utf-8") as file:
            json.dump(selected_summary, file, indent=4)

        for artifact_path in [
            selected_summary_path,
            Path(args.feature_importance_dir) / "all_feature_importance.csv",
            Path(args.feature_importance_dir) / "top_10_feature_importance.csv",
            Path(args.feature_importance_dir) / "top_10_features.json",
            feature_selection_results_path,
            selected_features_path,
            Path(args.report_dir) / "data_leakage_report.json",
        ]:
            if artifact_path.exists():
                mlflow.log_artifact(str(artifact_path))

        mlflow.sklearn.log_model(
            sk_model=selected_feature_pipeline,
            artifact_path="model",
        )

    selected_feature_model_path = Path(args.model_dir) / "best_selected_feature_model.pkl"
    joblib.dump(selected_feature_pipeline, selected_feature_model_path)

    run_summaries.append(
        {
            "model_name": f"{best_all_feature_model_name}_selected_features",
            "model_stage": "selected_features_analysis",
            "primary_metric": primary_metric,
            "score": selected_score,
            "test_score": selected_score,
            "feature_selection_used": True,
            "quality_verdict": selected_diagnostics["verdict"],
            "diagnostic_warning": " | ".join(selected_diagnostics["warnings"]),
            **{f"train_{k}": v for k, v in selected_train_metrics.items()},
            **{f"test_{k}": v for k, v in selected_test_metrics.items()},
        }
    )

    selected_is_better = (
        (higher_is_better and selected_score > best_all_feature_score)
        or ((not higher_is_better) and selected_score < best_all_feature_score)
    )

    if selected_is_better:
        final_model_choice = "selected_features"
        final_model_name = f"{best_all_feature_model_name}_selected_features"
        final_model = selected_feature_pipeline
        final_features = selected_features
        final_feature_selection_used = True
        final_reason = (
            "Selected-feature model outperformed the all-feature model on the primary metric."
        )
    else:
        final_model_choice = "all_features"
        final_model_name = f"{best_all_feature_model_name}_all_features"
        final_model = best_all_feature_model
        final_features = X_train.columns.tolist()
        final_feature_selection_used = False
        final_reason = (
            "Selected-feature model did not outperform the all-feature model, "
            "so feature importance is kept as analysis only."
        )

    if final_feature_selection_used:
        final_X_train_for_eval = X_train[final_features]
        final_X_test_for_eval = X_test[final_features]
    else:
        final_X_train_for_eval = X_train
        final_X_test_for_eval = X_test

    final_train_metrics = evaluate_model(
        model=final_model,
        X_test=final_X_train_for_eval,
        y_test=y_train,
        output_dir=args.report_dir,
    )

    final_test_metrics = evaluate_model(
        model=final_model,
        X_test=final_X_test_for_eval,
        y_test=y_test,
        output_dir=args.report_dir,
    )

    final_diagnostics = evaluate_overfit_underfit(
        model_name=final_model_name,
        train_metrics=final_train_metrics,
        test_metrics=final_test_metrics,
        problem_type=problem_type,
        primary_metric=primary_metric,
        higher_is_better=higher_is_better,
    )

    with mlflow.start_run(run_name=f"final_{final_model_name}"):
        mlflow.log_param("model_name", final_model_name)
        mlflow.log_param("model_stage", "final_model")
        mlflow.log_param("final_model_choice", final_model_choice)
        mlflow.log_param("final_reason", final_reason)
        mlflow.log_param("reference_best_all_feature_model_name", best_all_feature_model_name)
        mlflow.log_param("problem_type", problem_type)
        mlflow.log_param("target_column", args.target_column)
        mlflow.log_param("primary_metric", primary_metric)
        mlflow.log_param("all_features_score", float(best_all_feature_score))
        mlflow.log_param("selected_features_score", float(selected_score))
        mlflow.log_param("feature_selection_used", str(final_feature_selection_used).lower())
        mlflow.log_param("final_features", json.dumps(final_features))
        mlflow.log_param("selected_features_analysis", json.dumps(selected_features))
        mlflow.log_param("top_10_reference_features", json.dumps(top_10_payload["top_10_features"]))
        mlflow.log_param("quality_verdict", final_diagnostics["verdict"])
        mlflow.log_param("leakage_status", leakage_report["status"])
        mlflow.log_param("is_final_model", "true")

        for metric_name, metric_value in final_train_metrics.items():
            if isinstance(metric_value, (int, float)):
                mlflow.log_metric(f"final_train_{metric_name}", metric_value)

        for metric_name, metric_value in final_test_metrics.items():
            if isinstance(metric_value, (int, float)):
                mlflow.log_metric(f"final_test_{metric_name}", metric_value)
                mlflow.log_metric(metric_name, metric_value)

        if final_diagnostics["gap"] is not None:
            mlflow.log_metric("final_train_test_gap", final_diagnostics["gap"])

        final_run_id = mlflow.active_run().info.run_id

        for artifact_path in [
            Path(args.report_dir) / "metrics.json",
            Path(args.report_dir) / "classification_report.txt",
            Path(args.report_dir) / "confusion_matrix.csv",
            Path(args.report_dir) / "data_leakage_report.json",
            Path(args.feature_importance_dir) / "all_feature_importance.csv",
            Path(args.feature_importance_dir) / "top_10_feature_importance.csv",
            Path(args.feature_importance_dir) / "top_10_features.json",
            feature_selection_results_path,
            selected_features_path,
        ]:
            if artifact_path.exists():
                mlflow.log_artifact(str(artifact_path))

        mlflow.sklearn.log_model(
            sk_model=final_model,
            artifact_path="model",
        )

    best_model_path = Path(args.model_dir) / "best_model.pkl"
    joblib.dump(final_model, best_model_path)

    feature_metadata = {
        "target_column": args.target_column,
        "problem_type": problem_type,
        "final_model_choice": final_model_choice,
        "final_model_name": final_model_name,
        "feature_selection_used": final_feature_selection_used,
        "final_features": final_features,
        "selected_features_analysis": selected_features,
        "top_10_reference_features": top_10_payload["top_10_features"],
        "feature_importance_method": "permutation_importance",
        "feature_selection_method": "validation_cutoff_selection",
        "all_input_columns": X_train.columns.tolist(),
        "final_run_id": final_run_id,
        "model_path": str(best_model_path),
    }

    feature_metadata_path = Path(args.model_dir) / "feature_columns.json"

    with open(feature_metadata_path, "w", encoding="utf-8") as file:
        json.dump(feature_metadata, file, indent=4)

    summary = {
        "final_model_choice": final_model_choice,
        "final_model_name": final_model_name,
        "final_reason": final_reason,
        "primary_metric": primary_metric,
        "all_features_best_model_name": best_all_feature_model_name,
        "all_features_score": float(best_all_feature_score),
        "all_features_metrics": best_all_feature_test_metrics,
        "selected_features_score": float(selected_score),
        "selected_features_metrics": selected_test_metrics,
        "feature_importance_kept_as_analysis": not selected_is_better,
        "selected_features_analysis": selected_features,
        "top_10_reference_features": top_10_payload["top_10_features"],
        "final_features": final_features,
        "final_score": float(final_test_metrics[primary_metric]),
        "final_metrics": final_test_metrics,
        "model_path": str(best_model_path),
        "best_all_features_model_path": str(best_all_features_model_path),
        "selected_feature_model_path": str(selected_feature_model_path),
        "final_run_id": final_run_id,
        "leakage_status": leakage_report["status"],
    }

    summary_path = Path(args.report_dir) / "best_model_summary.json"

    with open(summary_path, "w", encoding="utf-8") as file:
        json.dump(summary, file, indent=4)

    run_summaries.append(
        {
            "model_name": final_model_name,
            "model_stage": "final_model",
            "primary_metric": primary_metric,
            "score": final_test_metrics[primary_metric],
            "test_score": final_test_metrics[primary_metric],
            "feature_selection_used": final_feature_selection_used,
            "quality_verdict": final_diagnostics["verdict"],
            "diagnostic_warning": " | ".join(final_diagnostics["warnings"]),
            **{f"train_{k}": v for k, v in final_train_metrics.items()},
            **{f"test_{k}": v for k, v in final_test_metrics.items()},
        }
    )

    run_summary_path = Path(args.report_dir) / "mlflow_run_summary.csv"
    pd.DataFrame(run_summaries).to_csv(run_summary_path, index=False)

    save_model_diagnostics_report(
        diagnostics=diagnostic_summaries,
        output_dir=args.report_dir,
        filename="model_diagnostics_report.json",
    )

    print("MLflow experiment logging completed.")
    print(json.dumps(summary, indent=4))

    print("\nTop 10 reference features:")
    for feature in top_10_payload["top_10_features"]:
        print(f"- {feature}")

    print("\nSelected features tested through feature importance:")
    for feature in selected_features:
        print(f"- {feature}")

    print("\nFinal model decision:")
    print(final_reason)

    print("\nModel quality diagnostics:")
    for item in diagnostic_summaries:
        print(f"- {item['model_name']}: {item['verdict']} | {' | '.join(item['warnings'])}")

    if leakage_report["warnings"]:
        print("\nData leakage warnings:")
        for warning in leakage_report["warnings"]:
            print(f"- {warning}")


def parse_args():
    """
    Parse command line arguments.
    """
    parser = argparse.ArgumentParser()

    parser.add_argument("--train-path", default="data/splits/train.csv")
    parser.add_argument("--test-path", default="data/splits/test.csv")
    parser.add_argument("--target-column", default="target")
    parser.add_argument("--model-dir", default="models")
    parser.add_argument("--report-dir", default="reports")
    parser.add_argument("--feature-importance-dir", default="reports/feature_importance")
    parser.add_argument("--experiment-name", default="mlops_training_experiments")
    parser.add_argument("--backend-store-path", default="mlruns/mlflow.db")
    parser.add_argument("--random-state", type=int, default=42)
    parser.add_argument("--validation-size", type=float, default=0.20)
    parser.add_argument("--permutation-repeats", type=int, default=8)

    return parser.parse_args()


if __name__ == "__main__":
    train_with_mlflow(parse_args())
'''

write_file("src/training/train.py", mlflow_train_code)

print("\nRunning MLflow experiments for Issue #20...")
run_and_print(
    [
        sys.executable,
        "src/training/train.py",
        "--train-path",
        "data/splits/train.csv",
        "--test-path",
        "data/splits/test.csv",
        "--target-column",
        TARGET_COLUMN,
    ]
)

if Path("reports/mlflow_run_summary.csv").exists():
    print("\nMLflow experiment comparison:")
    print(pd.read_csv("reports/mlflow_run_summary.csv").to_string(index=False))

if Path("reports/feature_importance/top_10_feature_importance.csv").exists():
    print("\nTop 10 feature importance:")
    print(pd.read_csv("reports/feature_importance/top_10_feature_importance.csv").to_string(index=False))

if Path("reports/feature_importance/selected_features.json").exists():
    print("\nSelected feature subset:")
    with open("reports/feature_importance/selected_features.json", "r", encoding="utf-8") as file:
        selected_features_summary = json.load(file)
    print(json.dumps(selected_features_summary, indent=4))

make_issue_zip(
    "issue_20_log_experiments.zip",
    [
        "src/training/train.py",
    ],
)

Created/updated: src/training/train.py

Running MLflow experiments for Issue #20...

COMMAND:
['/usr/bin/python3', 'src/training/train.py', '--train-path', 'data/splits/train.csv', '--test-path', 'data/splits/test.csv', '--target-column', 'target']

STDOUT:
MLflow experiment logging completed.
{
    "final_model_choice": "all_features",
    "final_model_name": "xgboost_all_features",
    "final_reason": "Selected-feature model did not outperform the all-feature model, so feature importance is kept as analysis only.",
    "primary_metric": "f1_macro",
    "all_features_best_model_name": "xgboost",
    "all_features_score": 0.9499949994999499,
    "all_features_metrics": {
        "problem_type": "classification",
        "accuracy": 0.95,
        "precision_macro": 0.9501800720288115,
        "recall_macro": 0.95,
        "f1_macro": 0.9499949994999499,
        "roc_auc": 0.9892000000000001
    },
    "selected_features_score": 0.9499949994999499,
    "selected_features_metrics": {
    

PosixPath('exports/issue_20_log_experiments.zip')

### **Issue #21 - Hyperparameter Tuning**

In [17]:
# ============================================================
# Issue #21 — Hyperparameter Tuning
# ============================================================
#
# Description:
# Tune stronger candidate models.
#
# Tasks:
# - Tune Random Forest.
# - Tune XGBoost.
# - Tune LightGBM.
# - Tune CatBoost.
# - Use RandomizedSearchCV.
# - Log HPO runs to MLflow.
# - Save tuned model artifacts.
# - Save HPO summaries.
# - Add overfitting / underfitting diagnostics.
#
# Files:
# - src/training/hpo.py
# ============================================================

hpo_code = r'''
# =========================
# Hyperparameter tuning with MLflow and diagnostics
# =========================

import argparse
import importlib
import json
from pathlib import Path

import joblib
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.evaluation.diagnostics import evaluate_overfit_underfit, save_model_diagnostics_report
from src.evaluation.evaluate import detect_problem_type, evaluate_model
from src.training.mlflow_setup import configure_mlflow


def optional_import(module_name, class_name):
    """
    Import optional HPO model classes safely.
    """
    try:
        module = importlib.import_module(module_name)
        return getattr(module, class_name)
    except Exception as error:
        print(f"Optional HPO model skipped: {class_name} from {module_name}. Reason: {error}")
        return None


def make_one_hot_encoder():
    """
    Create OneHotEncoder with compatibility across sklearn versions.
    """
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def load_split_data(train_path, test_path, target_column):
    """
    Load train/test split files and separate features from target.
    """
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    if target_column not in train_df.columns:
        raise ValueError(f"Target column '{target_column}' was not found in train data.")

    if target_column not in test_df.columns:
        raise ValueError(f"Target column '{target_column}' was not found in test data.")

    X_train = train_df.drop(columns=[target_column])
    y_train = train_df[target_column]

    X_test = test_df.drop(columns=[target_column])
    y_test = test_df[target_column]

    return X_train, y_train, X_test, y_test


def build_preprocessor(X_train):
    """
    Build preprocessing for numeric and categorical features.
    """
    numeric_features = X_train.select_dtypes(
        include=["int64", "float64", "int32", "float32"]
    ).columns.tolist()

    categorical_features = X_train.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", make_one_hot_encoder()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_features),
            ("categorical", categorical_pipeline, categorical_features),
        ]
    )

    return preprocessor


def get_hpo_candidates(problem_type, random_state):
    """
    Return HPO candidates and parameter grids.
    """
    candidates = {}

    if problem_type == "classification":
        candidates["random_forest"] = {
            "estimator": RandomForestClassifier(random_state=random_state, n_jobs=-1),
            "params": {
                "model__n_estimators": [100, 200, 300, 500],
                "model__max_depth": [None, 5, 10, 20, 30],
                "model__min_samples_split": [2, 5, 10],
                "model__min_samples_leaf": [1, 2, 4],
                "model__max_features": ["sqrt", "log2", None],
            },
        }

        XGBClassifier = optional_import("xgboost", "XGBClassifier")
        LGBMClassifier = optional_import("lightgbm", "LGBMClassifier")
        CatBoostClassifier = optional_import("catboost", "CatBoostClassifier")

        if XGBClassifier is not None:
            candidates["xgboost"] = {
                "estimator": XGBClassifier(
                    eval_metric="logloss",
                    random_state=random_state,
                ),
                "params": {
                    "model__n_estimators": [100, 200, 300],
                    "model__learning_rate": [0.01, 0.05, 0.1],
                    "model__max_depth": [3, 4, 5],
                    "model__subsample": [0.8, 0.9, 1.0],
                    "model__colsample_bytree": [0.8, 0.9, 1.0],
                },
            }

        if LGBMClassifier is not None:
            candidates["lightgbm"] = {
                "estimator": LGBMClassifier(
                    random_state=random_state,
                    verbosity=-1,
                ),
                "params": {
                    "model__n_estimators": [100, 200, 300],
                    "model__learning_rate": [0.01, 0.05, 0.1],
                    "model__num_leaves": [15, 31, 63],
                    "model__max_depth": [-1, 3, 5, 7],
                    "model__subsample": [0.8, 0.9, 1.0],
                },
            }

        if CatBoostClassifier is not None:
            candidates["catboost"] = {
                "estimator": CatBoostClassifier(
                    random_seed=random_state,
                    verbose=False,
                ),
                "params": {
                    "model__iterations": [100, 200, 300],
                    "model__learning_rate": [0.01, 0.05, 0.1],
                    "model__depth": [3, 4, 5, 6],
                    "model__l2_leaf_reg": [1, 3, 5, 7],
                },
            }

        scoring = "f1_macro"
        primary_metric = "f1_macro"
        higher_is_better = True

    else:
        candidates["random_forest"] = {
            "estimator": RandomForestRegressor(random_state=random_state, n_jobs=-1),
            "params": {
                "model__n_estimators": [100, 200, 300, 500],
                "model__max_depth": [None, 5, 10, 20, 30],
                "model__min_samples_split": [2, 5, 10],
                "model__min_samples_leaf": [1, 2, 4],
                "model__max_features": ["sqrt", "log2", None],
            },
        }

        XGBRegressor = optional_import("xgboost", "XGBRegressor")
        LGBMRegressor = optional_import("lightgbm", "LGBMRegressor")
        CatBoostRegressor = optional_import("catboost", "CatBoostRegressor")

        if XGBRegressor is not None:
            candidates["xgboost"] = {
                "estimator": XGBRegressor(random_state=random_state),
                "params": {
                    "model__n_estimators": [100, 200, 300],
                    "model__learning_rate": [0.01, 0.05, 0.1],
                    "model__max_depth": [3, 4, 5],
                    "model__subsample": [0.8, 0.9, 1.0],
                    "model__colsample_bytree": [0.8, 0.9, 1.0],
                },
            }

        if LGBMRegressor is not None:
            candidates["lightgbm"] = {
                "estimator": LGBMRegressor(random_state=random_state, verbosity=-1),
                "params": {
                    "model__n_estimators": [100, 200, 300],
                    "model__learning_rate": [0.01, 0.05, 0.1],
                    "model__num_leaves": [15, 31, 63],
                    "model__max_depth": [-1, 3, 5, 7],
                    "model__subsample": [0.8, 0.9, 1.0],
                },
            }

        if CatBoostRegressor is not None:
            candidates["catboost"] = {
                "estimator": CatBoostRegressor(random_seed=random_state, verbose=False),
                "params": {
                    "model__iterations": [100, 200, 300],
                    "model__learning_rate": [0.01, 0.05, 0.1],
                    "model__depth": [3, 4, 5, 6],
                    "model__l2_leaf_reg": [1, 3, 5, 7],
                },
            }

        scoring = "neg_root_mean_squared_error"
        primary_metric = "rmse"
        higher_is_better = False

    return candidates, scoring, primary_metric, higher_is_better


def run_hpo(args):
    """
    Run HPO across multiple candidate model families.
    """
    Path(args.model_dir).mkdir(parents=True, exist_ok=True)
    Path(args.report_dir).mkdir(parents=True, exist_ok=True)

    configure_mlflow(
        experiment_name=args.experiment_name,
        backend_store_path=args.backend_store_path,
    )

    X_train, y_train, X_test, y_test = load_split_data(
        train_path=args.train_path,
        test_path=args.test_path,
        target_column=args.target_column,
    )

    problem_type = detect_problem_type(y_train)

    candidates, scoring, primary_metric, higher_is_better = get_hpo_candidates(
        problem_type=problem_type,
        random_state=args.random_state,
    )

    best_model = None
    best_model_name = None
    best_score = None
    best_summary = None
    hpo_summaries = []
    diagnostics_summaries = []

    for model_name, candidate in candidates.items():
        preprocessor = build_preprocessor(X_train)

        pipeline = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("model", candidate["estimator"]),
            ]
        )

        search = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=candidate["params"],
            n_iter=args.n_iter,
            scoring=scoring,
            cv=args.cv,
            n_jobs=-1,
            random_state=args.random_state,
            verbose=1,
            return_train_score=True,
        )

        with mlflow.start_run(run_name=f"hpo_{model_name}"):
            search.fit(X_train, y_train)

            tuned_model = search.best_estimator_

            train_metrics = evaluate_model(
                model=tuned_model,
                X_test=X_train,
                y_test=y_train,
                output_dir=args.report_dir,
            )

            test_metrics = evaluate_model(
                model=tuned_model,
                X_test=X_test,
                y_test=y_test,
                output_dir=args.report_dir,
            )

            diagnostics = evaluate_overfit_underfit(
                model_name=f"hpo_{model_name}",
                train_metrics=train_metrics,
                test_metrics=test_metrics,
                problem_type=problem_type,
                primary_metric=primary_metric,
                higher_is_better=higher_is_better,
            )

            diagnostics_summaries.append(diagnostics)

            test_score = test_metrics[primary_metric]

            cv_results_path = Path(args.report_dir) / f"hpo_cv_results_{model_name}.csv"
            pd.DataFrame(search.cv_results_).to_csv(cv_results_path, index=False)

            model_path = Path(args.model_dir) / f"tuned_{model_name}_model.pkl"
            joblib.dump(tuned_model, model_path)

            summary = {
                "model_name": model_name,
                "problem_type": problem_type,
                "scoring": scoring,
                "primary_metric": primary_metric,
                "best_cv_score": float(search.best_score_),
                "test_score": float(test_score),
                "best_params": search.best_params_,
                "train_metrics": train_metrics,
                "test_metrics": test_metrics,
                "diagnostics": diagnostics,
                "model_path": str(model_path),
            }

            summary_path = Path(args.report_dir) / f"hpo_summary_{model_name}.json"

            with open(summary_path, "w", encoding="utf-8") as file:
                json.dump(summary, file, indent=4)

            mlflow.log_param("model_name", f"hpo_{model_name}")
            mlflow.log_param("problem_type", problem_type)
            mlflow.log_param("scoring", scoring)
            mlflow.log_param("cv", args.cv)
            mlflow.log_param("n_iter", args.n_iter)
            mlflow.log_param("quality_verdict", diagnostics["verdict"])

            for param_name, param_value in search.best_params_.items():
                mlflow.log_param(param_name, param_value)

            mlflow.log_metric("best_cv_score", float(search.best_score_))

            for metric_name, metric_value in train_metrics.items():
                if isinstance(metric_value, (int, float)):
                    mlflow.log_metric(f"train_{metric_name}", metric_value)

            for metric_name, metric_value in test_metrics.items():
                if isinstance(metric_value, (int, float)):
                    mlflow.log_metric(f"test_{metric_name}", metric_value)
                    mlflow.log_metric(metric_name, metric_value)

            if diagnostics["gap"] is not None:
                mlflow.log_metric("train_test_gap", diagnostics["gap"])

            mlflow.log_artifact(str(summary_path))
            mlflow.log_artifact(str(cv_results_path))
            mlflow.log_artifact(str(model_path))

            mlflow.sklearn.log_model(
                sk_model=tuned_model,
                artifact_path="model",
            )

            hpo_summaries.append(summary)

            should_replace = (
                best_score is None
                or (higher_is_better and test_score > best_score)
                or (not higher_is_better and test_score < best_score)
            )

            if should_replace:
                best_score = test_score
                best_model = tuned_model
                best_model_name = model_name
                best_summary = summary

    final_model_path = Path(args.model_dir) / "best_tuned_model.pkl"
    joblib.dump(best_model, final_model_path)

    final_summary = {
        "best_hpo_model_name": best_model_name,
        "primary_metric": primary_metric,
        "best_score": float(best_score),
        "best_model_path": str(final_model_path),
        "best_summary": best_summary,
        "all_hpo_summaries": hpo_summaries,
    }

    final_summary_path = Path(args.report_dir) / "hpo_summary.json"

    with open(final_summary_path, "w", encoding="utf-8") as file:
        json.dump(final_summary, file, indent=4)

    save_model_diagnostics_report(
        diagnostics=diagnostics_summaries,
        output_dir=args.report_dir,
        filename="hpo_diagnostics_report.json",
    )

    print("Hyperparameter tuning completed.")
    print(json.dumps(final_summary, indent=4))


def parse_args():
    """
    Parse command line arguments.
    """
    parser = argparse.ArgumentParser()

    parser.add_argument("--train-path", default="data/splits/train.csv")
    parser.add_argument("--test-path", default="data/splits/test.csv")
    parser.add_argument("--target-column", default="target")
    parser.add_argument("--model-dir", default="models")
    parser.add_argument("--report-dir", default="reports")
    parser.add_argument("--experiment-name", default="mlops_training_experiments")
    parser.add_argument("--backend-store-path", default="mlruns/mlflow.db")
    parser.add_argument("--random-state", type=int, default=42)
    parser.add_argument("--cv", type=int, default=3)
    parser.add_argument("--n-iter", type=int, default=6)

    return parser.parse_args()


if __name__ == "__main__":
    run_hpo(parse_args())
'''

write_file("src/training/hpo.py", hpo_code)

print("\nRunning hyperparameter tuning for Issue #21...")
run_and_print(
    [
        sys.executable,
        "src/training/hpo.py",
        "--train-path",
        "data/splits/train.csv",
        "--test-path",
        "data/splits/test.csv",
        "--target-column",
        TARGET_COLUMN,
        "--n-iter",
        "6",
        "--cv",
        "3",
    ]
)

if Path("reports/hpo_summary.json").exists():
    print("\nHPO summary:")
    with open("reports/hpo_summary.json", "r", encoding="utf-8") as file:
        hpo_summary = json.load(file)
    print(json.dumps(hpo_summary, indent=4))

make_issue_zip(
    "issue_21_hyperparameter_tuning.zip",
    [
        "src/training/hpo.py",
    ],
)

Created/updated: src/training/hpo.py

Running hyperparameter tuning for Issue #21...

COMMAND:
['/usr/bin/python3', 'src/training/hpo.py', '--train-path', 'data/splits/train.csv', '--test-path', 'data/splits/test.csv', '--target-column', 'target', '--n-iter', '6', '--cv', '3']

STDOUT:
Fitting 3 folds for each of 6 candidates, totalling 18 fits
Fitting 3 folds for each of 6 candidates, totalling 18 fits
Fitting 3 folds for each of 6 candidates, totalling 18 fits
Fitting 3 folds for each of 6 candidates, totalling 18 fits
Hyperparameter tuning completed.
{
    "best_hpo_model_name": "xgboost",
    "primary_metric": "f1_macro",
    "best_score": 0.9499949994999499,
    "best_model_path": "models/best_tuned_model.pkl",
    "best_summary": {
        "model_name": "xgboost",
        "problem_type": "classification",
        "scoring": "f1_macro",
        "primary_metric": "f1_macro",
        "best_cv_score": 0.917427076065831,
        "test_score": 0.9499949994999499,
        "best_params":

PosixPath('exports/issue_21_hyperparameter_tuning.zip')

### **Issue #22 - Register Best Model**

In [18]:
# ============================================================
# Issue #22 — Register Best Model
# ============================================================
#
# Description:
# Register the best-performing MLflow run.
#
# Tasks:
# - Search MLflow runs using the chosen metric.
# - Prefer highest F1 for classification.
# - Register the best model.
# - Save feature-importance analysis as registry metadata if available.
# - Save model registry summary as a JSON artifact.
# - Promote model using classic stages or aliases.
#
# Important:
# - Feature importance is not forced to be final.
# - The registry should use the best-performing model overall.
# - If a final model run ties with another run, prefer the run marked is_final_model=true.
# - reports/model_registry_summary.json records the registered MLflow model version.
#
# Files:
# - src/training/register_model.py
# ============================================================

register_model_code = r'''
# =========================
# Register best model in MLflow
# =========================

import argparse
import json
from pathlib import Path

import mlflow
from mlflow.tracking import MlflowClient

from src.training.mlflow_setup import configure_mlflow


def find_best_run(experiment_name, metric_name, higher_is_better=True):
    """
    Find the best MLflow run based on the selected metric.

    Logic:
    1. Search all runs that contain the selected metric.
    2. Sort by the metric.
    3. If tied, prefer the run marked is_final_model=true.
    """
    client = MlflowClient()

    experiment = client.get_experiment_by_name(experiment_name)

    if experiment is None:
        raise ValueError(
            f"Experiment '{experiment_name}' does not exist. "
            "Run training or HPO before registering the model."
        )

    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        max_results=1000,
    )

    valid_runs = []

    for run in runs:
        if metric_name in run.data.metrics:
            valid_runs.append(run)

    if not valid_runs:
        available_metrics = sorted(
            {
                metric
                for run in runs
                for metric in run.data.metrics.keys()
            }
        )

        raise ValueError(
            f"No MLflow runs found with metric '{metric_name}'. "
            f"Available metrics are: {available_metrics}"
        )

    valid_runs = sorted(
        valid_runs,
        key=lambda run: run.data.metrics[metric_name],
        reverse=higher_is_better,
    )

    best_metric_value = valid_runs[0].data.metrics[metric_name]

    tied_best_runs = [
        run
        for run in valid_runs
        if run.data.metrics[metric_name] == best_metric_value
    ]

    final_model_runs = [
        run
        for run in tied_best_runs
        if run.data.params.get("is_final_model") == "true"
    ]

    if final_model_runs:
        return final_model_runs[0]

    return valid_runs[0]


def load_json_if_exists(path):
    """
    Load a JSON file if it exists.
    """
    path = Path(path)

    if not path.exists():
        return {}

    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def save_registry_summary(
    args,
    result,
    run_id,
    best_metric_value,
    best_run,
    final_model_choice,
    final_reason,
    feature_importance_kept_as_analysis,
    selected_features,
    top_10_reference_features,
    promotion_message,
):
    """
    Save a local JSON summary of the MLflow registration result.

    This file is useful for DVC because it records:
    - registered model name
    - registered model version
    - run ID
    - metric value
    - final model choice
    - feature importance metadata
    """
    registry_summary = {
        "registered_model_name": args.registered_model_name,
        "registered_model_version": str(result.version),
        "registered_run_id": run_id,
        "metric_name": args.metric_name,
        "metric_value": float(best_metric_value),
        "model_name_from_run": best_run.data.params.get("model_name", "unknown"),
        "model_stage_from_run": best_run.data.params.get("model_stage", "unknown"),
        "final_model_choice": final_model_choice,
        "final_reason": final_reason,
        "feature_importance_kept_as_analysis": feature_importance_kept_as_analysis,
        "selected_features_analysis": selected_features,
        "top_10_reference_features": top_10_reference_features,
        "promotion_message": promotion_message,
    }

    summary_path = Path(args.registry_summary_path)
    summary_path.parent.mkdir(parents=True, exist_ok=True)

    with open(summary_path, "w", encoding="utf-8") as file:
        json.dump(registry_summary, file, indent=4)

    return summary_path, registry_summary


def register_and_promote_model(args):
    """
    Register the best MLflow model and promote it.

    Feature importance metadata is stored if available, but the registered model
    is chosen by best metric performance.
    """
    configure_mlflow(
        experiment_name=args.experiment_name,
        backend_store_path=args.backend_store_path,
    )

    best_run = find_best_run(
        experiment_name=args.experiment_name,
        metric_name=args.metric_name,
        higher_is_better=args.higher_is_better,
    )

    run_id = best_run.info.run_id
    best_metric_value = best_run.data.metrics[args.metric_name]

    model_uri = f"runs:/{run_id}/model"

    result = mlflow.register_model(
        model_uri=model_uri,
        name=args.registered_model_name,
    )

    client = MlflowClient()

    selected_features_payload = load_json_if_exists(args.selected_features_path)
    best_model_summary = load_json_if_exists(args.best_model_summary_path)

    selected_features = selected_features_payload.get("selected_features", [])
    top_10_reference_features = selected_features_payload.get("top_10_reference_features", [])

    final_model_choice = best_model_summary.get(
        "final_model_choice",
        best_run.data.params.get("final_model_choice", "best_metric_run"),
    )

    final_reason = best_model_summary.get(
        "final_reason",
        "Registered the run with the best selected metric.",
    )

    feature_importance_kept_as_analysis = best_model_summary.get(
        "feature_importance_kept_as_analysis",
        "unknown",
    )

    model_version_tags = {
        "registered_metric_name": args.metric_name,
        "registered_metric_value": str(best_metric_value),
        "registered_run_id": run_id,
        "registered_model_name_from_run": best_run.data.params.get("model_name", "unknown"),
        "registered_model_stage_from_run": best_run.data.params.get("model_stage", "unknown"),
        "final_model_choice": str(final_model_choice),
        "final_reason": str(final_reason),
        "feature_importance_kept_as_analysis": str(feature_importance_kept_as_analysis),
        "selected_features_analysis": json.dumps(selected_features),
        "top_10_reference_features": json.dumps(top_10_reference_features),
        "feature_importance_method": "permutation_importance",
        "feature_selection_method": "validation_cutoff_selection",
    }

    for tag_key, tag_value in model_version_tags.items():
        client.set_model_version_tag(
            name=args.registered_model_name,
            version=result.version,
            key=tag_key,
            value=tag_value,
        )

    promotion_message = ""

    try:
        client.transition_model_version_stage(
            name=args.registered_model_name,
            version=result.version,
            stage="Staging",
            archive_existing_versions=True,
        )

        client.transition_model_version_stage(
            name=args.registered_model_name,
            version=result.version,
            stage="Production",
            archive_existing_versions=True,
        )

        promotion_message = "Model moved to Production."

    except Exception as error:
        client.set_registered_model_alias(
            name=args.registered_model_name,
            alias="staging",
            version=result.version,
        )

        client.set_registered_model_alias(
            name=args.registered_model_name,
            alias="production",
            version=result.version,
        )

        promotion_message = (
            "Classic stage transition was not available. "
            "Aliases 'staging' and 'production' were set instead."
        )

        print(f"Stage transition warning: {error}")

    registry_summary_path, registry_summary = save_registry_summary(
        args=args,
        result=result,
        run_id=run_id,
        best_metric_value=best_metric_value,
        best_run=best_run,
        final_model_choice=final_model_choice,
        final_reason=final_reason,
        feature_importance_kept_as_analysis=feature_importance_kept_as_analysis,
        selected_features=selected_features,
        top_10_reference_features=top_10_reference_features,
        promotion_message=promotion_message,
    )

    print("Best model registered successfully.")
    print(f"Best run ID: {run_id}")
    print(f"Metric used: {args.metric_name}")
    print(f"Metric value: {best_metric_value}")
    print(f"Registered model name: {args.registered_model_name}")
    print(f"Registered model version: {result.version}")
    print(f"Model name from run: {best_run.data.params.get('model_name', 'unknown')}")
    print(f"Model stage from run: {best_run.data.params.get('model_stage', 'unknown')}")
    print(f"Final model choice: {final_model_choice}")
    print(f"Feature importance kept as analysis: {feature_importance_kept_as_analysis}")
    print(f"Selected features analysis: {selected_features}")
    print(f"Top 10 reference features: {top_10_reference_features}")
    print(f"Registry summary saved to: {registry_summary_path}")
    print(promotion_message)


def parse_args():
    """
    Parse command line arguments.
    """
    parser = argparse.ArgumentParser()

    parser.add_argument("--experiment-name", default="mlops_training_experiments")
    parser.add_argument("--backend-store-path", default="mlruns/mlflow.db")
    parser.add_argument("--registered-model-name", default="BestMLOpsModel")
    parser.add_argument("--metric-name", default="f1_macro")
    parser.add_argument("--higher-is-better", action="store_true")
    parser.add_argument("--selected-features-path", default="reports/feature_importance/selected_features.json")
    parser.add_argument("--best-model-summary-path", default="reports/best_model_summary.json")
    parser.add_argument("--registry-summary-path", default="reports/model_registry_summary.json")

    return parser.parse_args()


if __name__ == "__main__":
    register_and_promote_model(parse_args())
'''

write_file("src/training/register_model.py", register_model_code)

print("\nRunning model registration for Issue #22...")
result = subprocess.run(
    [
        sys.executable,
        "src/training/register_model.py",
        "--experiment-name",
        "mlops_training_experiments",
        "--registered-model-name",
        "BestMLOpsModel",
        "--metric-name",
        "f1_macro",
        "--higher-is-better",
        "--selected-features-path",
        "reports/feature_importance/selected_features.json",
        "--best-model-summary-path",
        "reports/best_model_summary.json",
        "--registry-summary-path",
        "reports/model_registry_summary.json",
    ],
    capture_output=True,
    text=True,
)

print("STDOUT:")
print(result.stdout)

print("STDERR:")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Model registration failed. See STDERR above.")

if Path("reports/model_registry_summary.json").exists():
    print("\nModel registry summary:")
    with open("reports/model_registry_summary.json", "r", encoding="utf-8") as file:
        model_registry_summary = json.load(file)
    print(json.dumps(model_registry_summary, indent=4))

make_issue_zip(
    "issue_22_register_best_model.zip",
    [
        "src/training/register_model.py",
    ],
)

print("Issue #22 exported successfully.")

Created/updated: src/training/register_model.py

Running model registration for Issue #22...
STDOUT:
Best model registered successfully.
Best run ID: 2a2b3120871e418aac2ab088ff50db95
Metric used: f1_macro
Metric value: 0.9499949994999499
Registered model name: BestMLOpsModel
Registered model version: 1
Model name from run: xgboost_all_features
Model stage from run: final_model
Final model choice: all_features
Feature importance kept as analysis: True
Selected features analysis: ['feature_1', 'feature_5', 'feature_3', 'feature_4', 'feature_6', 'feature_7', 'feature_0', 'feature_2']
Top 10 reference features: ['feature_1', 'feature_5', 'feature_3', 'feature_4', 'feature_6', 'feature_7', 'feature_0', 'feature_2']
Registry summary saved to: reports/model_registry_summary.json
Model moved to Production.

STDERR:
Successfully registered model 'BestMLOpsModel'.
2026/05/07 03:50:47 WARNING mlflow.tracking._model_registry.fluent: Run with id 2a2b3120871e418aac2ab088ff50db95 has no artifacts at 

### **Issue #16 - DVC Pipeline Stages**

In [19]:
# ============================================================
# Issue #16 — DVC Pipeline Stages
# ============================================================
#
# Description:
# Update the shared DVC pipeline.
#
# Shahd's responsibility:
# - prepare
# - features
# - preprocess
#
# Ibrahim's responsibility:
# - train
# - hpo
# - register
#
# Rules:
# - Do not create new prepare/preprocess/features scripts.
# - Do not replace Shahd's dvc.yaml from scratch.
# - Use Shahd's uploaded dvc.yaml as the base.
# - Append or update only Ibrahim's stages.
# - Do not run dvc pull in Colab.
# - In Colab, run only Ibrahim's DVC stages with --single-item:
#   - train
#   - hpo
#   - register
#
# Why --single-item is used:
# - The notebook manually uploads train.csv and test.csv.
# - We do not want Colab DVC to rerun Shahd's upstream data stages.
# - The full DVC pipeline can still be tested locally later with the full remote.
#
# Files exported:
# - dvc.yaml
# - dvc.lock, if DVC repro succeeds
# ============================================================

os.chdir(PROJECT_ROOT)
os.environ["PYTHONPATH"] = f"{PROJECT_ROOT}:{os.environ.get('PYTHONPATH', '')}"

print("Using target column:", TARGET_COLUMN)


# ============================================================
# Step 1 — Remove wrongly generated duplicate training-stage files
# ============================================================
#
# Purpose:
# - Make sure old wrong files are removed from the Colab workspace.
# - These files should not be committed because Shahd already owns these stages.
# ============================================================

duplicate_stage_files = [
    "src/training/prepare_stage.py",
    "src/training/preprocess_stage.py",
    "src/training/featurize_stage.py",
]

for duplicate_file in duplicate_stage_files:
    duplicate_path = Path(duplicate_file)

    if duplicate_path.exists():
        duplicate_path.unlink()
        print(f"Removed duplicated file: {duplicate_path}")


# ============================================================
# Step 2 — Validate existing shared data-stage scripts and config
# ============================================================
#
# Purpose:
# - Confirm that Shahd's prepare.py and preprocess.py exist.
# - Confirm that Shahd's params.yaml exists.
# - Confirm that the uploaded dvc.yaml exists.
# ============================================================

prepare_script = Path("src/data/prepare.py")
preprocess_script = Path("src/data/preprocess.py")
params_file = Path("configs/params.yaml")
dvc_yaml_file = Path("dvc.yaml")

if not prepare_script.exists():
    raise FileNotFoundError(
        "Expected existing data preparation script was not found: src/data/prepare.py."
    )

if not preprocess_script.exists():
    raise FileNotFoundError(
        "Expected existing preprocessing script was not found: src/data/preprocess.py."
    )

if not params_file.exists():
    raise FileNotFoundError(
        "Expected configs/params.yaml was not found. Upload Shahd's latest params.yaml."
    )

if not dvc_yaml_file.exists():
    raise FileNotFoundError(
        "Expected uploaded dvc.yaml was not found. Upload Shahd's latest dvc.yaml."
    )

print("Existing shared files found:")
print(f"- {prepare_script}")
print(f"- {preprocess_script}")
print(f"- {params_file}")
print("- dvc.yaml")


# ============================================================
# Step 3 — Read Shahd's existing dvc.yaml
# ============================================================
#
# Purpose:
# - Load the existing shared DVC pipeline.
# - Preserve Shahd's stages.
# - Append or update Ibrahim's train, hpo, and register stages.
# ============================================================

with open("dvc.yaml", "r", encoding="utf-8") as file:
    existing_dvc_yaml = yaml.safe_load(file)

if existing_dvc_yaml is None:
    existing_dvc_yaml = {}

if "stages" not in existing_dvc_yaml:
    existing_dvc_yaml["stages"] = {}

existing_stage_names = list(existing_dvc_yaml["stages"].keys())

print("\nExisting stages in uploaded dvc.yaml:")
for stage_name in existing_stage_names:
    print("-", stage_name)


# ============================================================
# Step 4 — Append Ibrahim's stages to dvc.yaml
# ============================================================
#
# Purpose:
# - Add train stage after Shahd's data stages.
# - Add hpo stage after train.
# - Add register stage after hpo.
# - Add model registry summary output so the registered MLflow version is recorded.
#
# Notes:
# - If train/hpo/register already exist, they are updated.
# - The existing prepare/features/preprocess stages are not replaced.
# - The register stage assumes register_model.py supports:
#   --registry-summary-path reports/model_registry_summary.json
# ============================================================

existing_dvc_yaml["stages"]["train"] = {
    "cmd": (
        f"python src/training/train.py "
        f"--train-path data/splits/train.csv "
        f"--test-path data/splits/test.csv "
        f"--target-column {TARGET_COLUMN} "
        f"--model-dir models "
        f"--report-dir reports"
    ),
    "deps": [
        "src/training/train.py",
        "src/training/mlflow_setup.py",
        "src/evaluation/evaluate.py",
        "src/evaluation/diagnostics.py",
        "data/splits/train.csv",
        "data/splits/test.csv",
    ],
    "outs": [
        "models/best_model.pkl",
        "models/best_reference_model.pkl",
        "models/best_all_features_model.pkl",
        "models/best_selected_feature_model.pkl",
        "models/feature_columns.json",
        "reports/best_model_summary.json",
        "reports/selected_feature_model_summary.json",
        "reports/mlflow_run_summary.csv",
        "reports/model_diagnostics_report.json",
        "reports/data_leakage_report.json",
        "reports/feature_importance/all_feature_importance.csv",
        "reports/feature_importance/top_10_feature_importance.csv",
        "reports/feature_importance/top_10_features.json",
        "reports/feature_importance/feature_selection_results.json",
        "reports/feature_importance/selected_features.json",
    ],
    "metrics": [
        {
            "reports/metrics.json": {
                "cache": False,
            }
        }
    ],
}

existing_dvc_yaml["stages"]["hpo"] = {
    "cmd": (
        f"python src/training/hpo.py "
        f"--train-path data/splits/train.csv "
        f"--test-path data/splits/test.csv "
        f"--target-column {TARGET_COLUMN} "
        f"--model-dir models "
        f"--report-dir reports "
        f"--n-iter 6 "
        f"--cv 3"
    ),
    "deps": [
        "src/training/hpo.py",
        "src/training/mlflow_setup.py",
        "src/evaluation/evaluate.py",
        "src/evaluation/diagnostics.py",
        "data/splits/train.csv",
        "data/splits/test.csv",
    ],
    "outs": [
        "models/best_tuned_model.pkl",
        "reports/hpo_summary.json",
        "reports/hpo_diagnostics_report.json",
    ],
}

existing_dvc_yaml["stages"]["register"] = {
    "cmd": (
        f"python src/training/register_model.py "
        f"--experiment-name mlops_training_experiments "
        f"--registered-model-name BestMLOpsModel "
        f"--metric-name f1_macro "
        f"--higher-is-better "
        f"--selected-features-path reports/feature_importance/selected_features.json "
        f"--best-model-summary-path reports/best_model_summary.json "
        f"--registry-summary-path reports/model_registry_summary.json"
    ),
    "deps": [
        "src/training/register_model.py",
        "src/training/mlflow_setup.py",
        "reports/feature_importance/selected_features.json",
        "reports/best_model_summary.json",
    ],
    "outs": [
        "reports/model_registry_summary.json",
    ],
    "always_changed": True,
}

with open("dvc.yaml", "w", encoding="utf-8") as file:
    yaml.safe_dump(
        existing_dvc_yaml,
        file,
        sort_keys=False,
        default_flow_style=False,
    )

print("\nUpdated dvc.yaml stages:")
for stage_name in existing_dvc_yaml["stages"].keys():
    print("-", stage_name)


# ============================================================
# Step 5 — DVC execution policy for Colab
# ============================================================
#
# Purpose:
# - Run DVC inside Colab without requiring GitHub or dvc pull.
# - Run only Ibrahim's stages because the train/test CSVs were uploaded manually.
# - Use --single-item so DVC does not try to reproduce Shahd's upstream stages.
#
# Important:
# - Running the register stage may create another MLflow model version.
# - The registered version should be saved to reports/model_registry_summary.json.
# ============================================================

RUN_DVC_REPRO = True

DVC_TARGET_STAGES = [
    "train",
    "hpo",
    "register",
]

dvc_results = {
    "dvc_init_success": False,
    "dvc_repro_success": False,
    "dvc_status_success": False,
    "dvc_metrics_success": False,
    "dvc_dag_success": False,
    "stages_attempted": DVC_TARGET_STAGES,
    "stage_results": {},
}


# ============================================================
# Step 6 — Temporarily disable raw .dvc pointer during Colab DVC run
# ============================================================
#
# Purpose:
# - The raw .dvc pointer may reference a raw CSV that is not available in Colab.
# - We are testing Ibrahim's stages only in Colab.
# - Temporarily disable the raw .dvc pointer so it does not interfere.
# - Restore it after the DVC test.
# ============================================================

raw_dvc_path = Path("data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv.dvc")
disabled_raw_dvc_path = Path("data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv.dvc.disabled_for_colab_dvc_test")

raw_dvc_was_disabled = False

if raw_dvc_path.exists():
    shutil.move(str(raw_dvc_path), str(disabled_raw_dvc_path))
    raw_dvc_was_disabled = True
    print("Temporarily disabled raw .dvc pointer for Colab DVC execution.")


# ============================================================
# Step 7 — Helper function for DVC commands
# ============================================================
#
# Purpose:
# - Run DVC commands using the active Python executable.
# - Capture stdout, stderr, and return code.
# - Save outputs into reports/pipeline for review.
# ============================================================

def run_dvc_command(command_args, label):
    """
    Run a DVC command and save its output.
    """
    reports_pipeline_dir = Path("reports/pipeline")
    reports_pipeline_dir.mkdir(parents=True, exist_ok=True)

    full_command = [sys.executable, "-m", "dvc"] + command_args

    result = subprocess.run(
        full_command,
        capture_output=True,
        text=True,
    )

    stdout_path = reports_pipeline_dir / f"dvc_{label}_stdout.txt"
    stderr_path = reports_pipeline_dir / f"dvc_{label}_stderr.txt"

    stdout_path.write_text(result.stdout, encoding="utf-8")
    stderr_path.write_text(result.stderr, encoding="utf-8")

    print("\nDVC COMMAND:")
    print(" ".join(full_command))

    print("\nDVC STDOUT:")
    print(result.stdout)

    print("\nDVC STDERR:")
    print(result.stderr)

    return {
        "label": label,
        "command": " ".join(full_command),
        "returncode": result.returncode,
        "stdout_path": str(stdout_path),
        "stderr_path": str(stderr_path),
        "success": result.returncode == 0,
    }


# ============================================================
# Step 8 — Initialize DVC in Colab workspace
# ============================================================
#
# Purpose:
# - Initialize DVC using --no-scm because Colab is not connected to GitHub.
# - If .dvc already exists, skip initialization.
# ============================================================

if RUN_DVC_REPRO:
    if not Path(".dvc").exists():
        init_result = run_dvc_command(["init", "--no-scm"], "init")
        dvc_results["dvc_init_success"] = init_result["success"]
        dvc_results["dvc_init_result"] = init_result

        if not init_result["success"]:
            print("DVC init failed. DVC repro will be skipped.")
            RUN_DVC_REPRO = False
    else:
        print("Existing DVC metadata found. Skipping dvc init.")
        dvc_results["dvc_init_success"] = True
else:
    print("Skipping dvc init because RUN_DVC_REPRO is False.")


# ============================================================
# Step 9 — Remove old dvc.lock before regenerating
# ============================================================
#
# Purpose:
# - Avoid using a stale dvc.lock from a previous run.
# - Regenerate lockfile from the corrected dvc.yaml.
# ============================================================

if RUN_DVC_REPRO:
    if Path("dvc.lock").exists():
        Path("dvc.lock").unlink()
        print("Removed old dvc.lock so it can be regenerated from corrected dvc.yaml.")
else:
    print("Skipping dvc.lock regeneration because RUN_DVC_REPRO is False.")


# ============================================================
# Step 10 — Run Ibrahim's DVC stages with --single-item
# ============================================================
#
# Purpose:
# - Run train stage.
# - Run hpo stage.
# - Run register stage.
# - Stop running later stages if one stage fails.
#
# Critical:
# - --single-item prevents DVC from trying to reproduce upstream stages.
# - This is what makes the Colab manual-upload workflow work correctly.
# ============================================================

if RUN_DVC_REPRO:
    all_stage_success = True

    for target_stage in DVC_TARGET_STAGES:
        stage_result = run_dvc_command(
            ["repro", "--single-item", target_stage],
            f"repro_{target_stage}",
        )

        dvc_results["stage_results"][target_stage] = stage_result

        if not stage_result["success"]:
            all_stage_success = False
            print(f"DVC stage failed: {target_stage}")
            print("Stopping DVC stage execution.")
            break

    dvc_results["dvc_repro_success"] = all_stage_success

else:
    print("DVC repro was skipped.")


# ============================================================
# Step 11 — Show DVC status, DAG, and metrics
# ============================================================
#
# Purpose:
# - Show DVC pipeline status.
# - Show DVC DAG.
# - Show DVC metrics.
# - Save outputs to reports/pipeline.
# ============================================================

if dvc_results["dvc_repro_success"]:
    status_result = run_dvc_command(["status"], "status")
    dag_result = run_dvc_command(["dag"], "dag")
    metrics_result = run_dvc_command(["metrics", "show"], "metrics_show")

    dvc_results["dvc_status_success"] = status_result["success"]
    dvc_results["dvc_dag_success"] = dag_result["success"]
    dvc_results["dvc_metrics_success"] = metrics_result["success"]

    dvc_results["dvc_status_result"] = status_result
    dvc_results["dvc_dag_result"] = dag_result
    dvc_results["dvc_metrics_result"] = metrics_result

else:
    print("Skipping DVC status, DAG, and metrics because DVC repro did not fully succeed.")


# ============================================================
# Step 12 — Restore raw .dvc pointer after Colab DVC run
# ============================================================
#
# Purpose:
# - Restore uploaded raw .dvc pointer after the DVC execution test.
# - The temporary rename is only used to avoid Colab-local raw data issues.
# ============================================================

if raw_dvc_was_disabled and disabled_raw_dvc_path.exists():
    shutil.move(str(disabled_raw_dvc_path), str(raw_dvc_path))
    print("Restored raw .dvc pointer after Colab DVC execution.")


# ============================================================
# Step 13 — Save DVC results summary
# ============================================================
#
# Purpose:
# - Save a structured summary of the DVC run.
# - Include whether dvc.lock was created.
# - Include whether model registry summary was created.
# ============================================================

dvc_results["dvc_lock_created"] = Path("dvc.lock").exists()
dvc_results["dvc_yaml_exists"] = Path("dvc.yaml").exists()
dvc_results["model_registry_summary_created"] = Path("reports/model_registry_summary.json").exists()

dvc_results_path = Path("reports/pipeline/dvc_results_summary.json")
dvc_results_path.parent.mkdir(parents=True, exist_ok=True)

with open(dvc_results_path, "w", encoding="utf-8") as file:
    json.dump(dvc_results, file, indent=4)

print("\nDVC results summary:")
print(json.dumps(dvc_results, indent=4))


# ============================================================
# Step 14 — Print MLflow registry summary created by DVC register stage
# ============================================================
#
# Purpose:
# - Show the registered MLflow model version.
# - Show the registered run ID.
# - Show the metric used.
# - This is what you can record for the project.
# ============================================================

print("\nModel registry summary:")
registry_summary_path = Path("reports/model_registry_summary.json")

if registry_summary_path.exists():
    with open(registry_summary_path, "r", encoding="utf-8") as file:
        registry_summary = json.load(file)
    print(json.dumps(registry_summary, indent=4))
else:
    print("reports/model_registry_summary.json was not created.")
    print("Make sure src/training/register_model.py supports --registry-summary-path.")


# ============================================================
# Step 15 — Export Issue #16 ZIP
# ============================================================
#
# Purpose:
# - Export dvc.yaml.
# - Export dvc.lock if DVC successfully created it.
# ============================================================

issue_16_files = [
    "dvc.yaml",
]

if Path("dvc.lock").exists():
    issue_16_files.append("dvc.lock")

make_issue_zip(
    "issue_16_dvc_pipeline.zip",
    issue_16_files,
)

print("\nIssue #16 exported successfully.")
print("Download this file from Colab:")
print("exports/issue_16_dvc_pipeline.zip")

Using target column: target
Existing shared files found:
- src/data/prepare.py
- src/data/preprocess.py
- configs/params.yaml
- dvc.yaml

Existing stages in uploaded dvc.yaml:
- prepare
- featurize
- preprocess

Updated dvc.yaml stages:
- prepare
- featurize
- preprocess
- train
- hpo
- register
Temporarily disabled raw .dvc pointer for Colab DVC execution.

DVC COMMAND:
/usr/bin/python3 -m dvc init --no-scm

DVC STDOUT:
Initialized DVC repository.

+---------------------------------------------------------------------+
|                                                                     |
|        DVC has enabled anonymous aggregate usage analytics.         |
|     Read the analytics documentation (and how to opt-out) here:     |
|             <https://dvc.org/doc/user-guide/analytics>              |
|                                                                     |
+---------------------------------------------------------------------+

What's next?
------------
- Check out the

### **Final Summary and Downloads**

In [20]:
# ============================================================
# Final summary and download section
# ============================================================
#
# Purpose:
# - Print final metrics.
# - Print best model summary.
# - Print feature importance analysis.
# - Print selected feature subset.
# - Print model diagnostics.
# - Print leakage diagnostics.
# - Print DVC results.
# - Generate backup ZIP.
# - Download all issue ZIPs from Colab.
# ============================================================

print("\nFinal metrics from reports/metrics.json:")
if Path("reports/metrics.json").exists():
    with open("reports/metrics.json", "r", encoding="utf-8") as file:
        final_metrics = json.load(file)
    print(json.dumps(final_metrics, indent=4))
else:
    print("reports/metrics.json was not found.")

print("\nBest model summary:")
if Path("reports/best_model_summary.json").exists():
    with open("reports/best_model_summary.json", "r", encoding="utf-8") as file:
        best_model_summary = json.load(file)
    print(json.dumps(best_model_summary, indent=4))
else:
    print("reports/best_model_summary.json was not found.")

print("\nTop 10 feature importance:")
if Path("reports/feature_importance/top_10_feature_importance.csv").exists():
    print(pd.read_csv("reports/feature_importance/top_10_feature_importance.csv").to_string(index=False))
else:
    print("reports/feature_importance/top_10_feature_importance.csv was not found.")

print("\nSelected feature subset:")
if Path("reports/feature_importance/selected_features.json").exists():
    with open("reports/feature_importance/selected_features.json", "r", encoding="utf-8") as file:
        selected_feature_report = json.load(file)
    print(json.dumps(selected_feature_report, indent=4))
else:
    print("reports/feature_importance/selected_features.json was not found.")

print("\nModel diagnostics report:")
if Path("reports/model_diagnostics_report.json").exists():
    with open("reports/model_diagnostics_report.json", "r", encoding="utf-8") as file:
        diagnostics_report = json.load(file)
    print(json.dumps(diagnostics_report, indent=4))
else:
    print("reports/model_diagnostics_report.json was not found.")

print("\nData leakage report:")
if Path("reports/data_leakage_report.json").exists():
    with open("reports/data_leakage_report.json", "r", encoding="utf-8") as file:
        leakage_report = json.load(file)
    print(json.dumps(leakage_report, indent=4))
else:
    print("reports/data_leakage_report.json was not found.")

print("\nDVC results report:")
if Path("reports/pipeline/dvc_results_summary.json").exists():
    with open("reports/pipeline/dvc_results_summary.json", "r", encoding="utf-8") as file:
        dvc_report = json.load(file)
    print(json.dumps(dvc_report, indent=4))
else:
    print("reports/pipeline/dvc_results_summary.json was not found.")

print("\nDVC metrics output:")
dvc_metrics_stdout_path = Path("reports/pipeline/dvc_metrics_show_stdout.txt")

if dvc_metrics_stdout_path.exists():
    print(dvc_metrics_stdout_path.read_text(encoding="utf-8"))
else:
    print("DVC metrics output was not found.")

print("\nDVC status output:")
dvc_status_stdout_path = Path("reports/pipeline/dvc_status_stdout.txt")

if dvc_status_stdout_path.exists():
    print(dvc_status_stdout_path.read_text(encoding="utf-8"))
else:
    print("DVC status output was not found.")

print("\nDVC DAG output:")
dvc_dag_stdout_path = Path("reports/pipeline/dvc_dag_stdout.txt")

if dvc_dag_stdout_path.exists():
    print(dvc_dag_stdout_path.read_text(encoding="utf-8"))
else:
    print("DVC DAG output was not found.")

print("\nGenerated issue ZIP files:")
for zip_file in sorted(Path("exports").glob("issue_*.zip")):
    print("-", zip_file)

make_issue_zip(
    "all_final_training_files_backup.zip",
    [
        "src/__init__.py",
        "src/data/__init__.py",
        "src/training/__init__.py",
        "src/training/mlflow_setup.py",
        "src/training/train.py",
        "src/training/hpo.py",
        "src/training/register_model.py",
        "src/evaluation/__init__.py",
        "src/evaluation/evaluate.py",
        "src/evaluation/diagnostics.py",
        "dvc.yaml",
        "dvc.lock",
    ],
)

try:
    from google.colab import files

    print("\nDownloading issue ZIP files...")
    for zip_file in sorted(Path("exports").glob("issue_*.zip")):
        files.download(str(zip_file))

    print("\nDownloading backup ZIP...")
    files.download("exports/all_final_training_files_backup.zip")

except Exception:
    print("\nDownload skipped because this is not running inside Colab.")

print("\nDone.")
print("Use the issue ZIP files for separate GitHub issue branches and PRs.")
print("Correct practical PR order:")
print("1. issue_19_setup_mlflow.zip")
print("2. issue_18_model_evaluation.zip")
print("3. issue_17_train_baseline.zip")
print("4. issue_20_log_experiments.zip")
print("5. issue_21_hyperparameter_tuning.zip")
print("6. issue_22_register_best_model.zip")
print("7. issue_16_dvc_pipeline.zip")


Final metrics from reports/metrics.json:
{
    "problem_type": "classification",
    "accuracy": 0.94,
    "precision_macro": 0.94,
    "recall_macro": 0.94,
    "f1_macro": 0.94,
    "roc_auc": 0.9863999999999999
}

Best model summary:
{
    "final_model_choice": "all_features",
    "final_model_name": "xgboost_all_features",
    "final_reason": "Selected-feature model did not outperform the all-feature model, so feature importance is kept as analysis only.",
    "primary_metric": "f1_macro",
    "all_features_best_model_name": "xgboost",
    "all_features_score": 0.9499949994999499,
    "all_features_metrics": {
        "problem_type": "classification",
        "accuracy": 0.95,
        "precision_macro": 0.9501800720288115,
        "recall_macro": 0.95,
        "f1_macro": 0.9499949994999499,
        "roc_auc": 0.9892000000000001
    },
    "selected_features_score": 0.9499949994999499,
    "selected_features_metrics": {
        "problem_type": "classification",
        "accuracy":

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Done.
Use the issue ZIP files for separate GitHub issue branches and PRs.
Correct practical PR order:
1. issue_19_setup_mlflow.zip
2. issue_18_model_evaluation.zip
3. issue_17_train_baseline.zip
4. issue_20_log_experiments.zip
5. issue_21_hyperparameter_tuning.zip
6. issue_22_register_best_model.zip
7. issue_16_dvc_pipeline.zip
